# 🪢 07 · Regularization: ridge, lasso and sparse genetic effects
### When there are more SNPs than people: MAP estimation, ridge regression, the lasso, soft-thresholding and coordinate descent

**BiologyToML** · Part B — Supervised learning · Module 07 of 24

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/07_ridge_lasso_sparse_regression.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.optimize`, `scipy.stats`), pandas, Matplotlib, scikit-learn (for checking only) |
| **Prerequisites** | Module 06 (linear regression, least squares, normal equations); Modules 03–05 (eigenvalues, gradients, MAP). |
| **Estimated time** | 150 min (reading + running + the ✏️ exercises) |
| **02-620 lectures covered** | `002_regression` slides 45–70; `015_model-selection--extra` slides 1–2 — see [`SLIDE_MAP.md`](../SLIDE_MAP.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The prostate-cancer measurements (Stamey *et al.* 1989; 97 men) are **real** data shipped with the course. All genotypes and phenotypes are **SIMULATED** (Hardy–Weinberg genotypes, with linkage-disequilibrium blocks in Sections 5, 9 and 10) so that we know which SNPs are truly causal. A selected SNP is never proof of a causal variant.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Derive** the MAP estimate of the regression weights under a Gaussian prior, step by step, and **show** that it is ridge regression $\mathbf w = (\mathbf X^\top\mathbf X + \lambda\mathbf I)^{-1}\mathbf X^\top\mathbf y$ with $\lambda = \sigma^2/\sigma_0^2$ (slides 47–57).
2. **Explain** why ridge works when $N<J$ (every eigenvalue is raised by $\lambda$) and how $\lambda$ trades bias for variance.
3. **Apply** the ridge recipe in practice: centre the data, leave the intercept unpenalised, standardise the features, and **choose** $\lambda$ on a validation set (slides 58–59).
4. **Explain** why sparsity makes biological and statistical sense, why the $L_0$ (best-subset) problem is intractable, and why the $L_1$ diamond produces exact zeros; **connect** the lasso to a Laplace prior (slides 61–68).
5. **Derive** the soft-thresholding solution of the one-dimensional lasso and **implement** pathwise coordinate descent from scratch, checked against scikit-learn (slide 69).
6. **Compare** univariate tests, ridge and lasso for finding causal SNPs in a $p \gg n$ GWAS with LD, and for predicting PSA in real prostate-cancer data.
7. **Interpret** selected SNPs with care: correlated SNPs, selection instability, the elastic net, and why selected ≠ causal.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit | 5 min |
| 1 · When least squares fails | recap of $N<J$; Bayes' rule for the weights (slides 45–48) | 10 min |
| 2 · MAP with a Gaussian prior = ridge | full derivation, numerical checks, invertibility (slides 49–54) | 20 min |
| 3 · Ridge as regularisation | $\lambda=\sigma^2/\sigma_0^2$, shrinkage, ridge path, bias vs variance (slides 55–57) | 15 min |
| 4 · Ridge in practice | centring, the unpenalised intercept, scaling (slides 58–59) | 15 min |
| 5 · Choosing $\lambda$ | a validation set on a $p\gg n$ GWAS cohort with LD | 10 min |
| 6 · Sparsity and the lasso | $L_0$ vs $L_1$, best subsets, geometry, Laplace prior (slides 60–68) | 20 min |
| 7 · Solving the lasso | soft-thresholding, coordinate descent, warm starts (slide 69) | 20 min |
| 8 · Prostate cancer again | lasso path, cross-validated ridge and lasso, test error | 10 min |
| 9 · GWAS with the lasso | univariate vs ridge vs lasso for finding causal SNPs (extra slides 1–2) | 10 min |
| 10 · Elastic net and caveats | correlated SNPs, instability, selected ≠ causal | 10 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 5 min |

## 🧫 The biological question

A modern GWAS measures **millions** of SNPs in thousands of people; a gene-expression study measures 20,000 genes in a
few dozen tumours. In both, there are far more features than samples: $J \gg N$. Module 06 ended with the bad news that
least squares then has infinitely many perfect — and useless — solutions.

Biology suggests a way out. Most SNPs have **no** effect on a given trait. Huntington's disease is caused by a single
gene, cystic fibrosis by mutations in one gene (*CFTR*), and much of the strongly inherited breast-cancer risk in
families by a few genes such as *BRCA1* and *BRCA2* (the examples of slide 62's notes); even for highly polygenic traits,
the effects of most variants are tiny. So instead of asking the data to determine
2,000 or 3 million numbers from scratch, we bring in **prior knowledge**: *effects are probably small* (ridge regression)
or *most effects are exactly zero* (the lasso). In the language of Module 05 this is MAP estimation; in the language of
statistics it is **regularisation**.

The lasso (Tibshirani 1996) turned "which SNPs matter?" into one convex optimisation problem and is now used throughout
genomics — for GWAS (Wu *et al.* 2009, cited on slides 63 and 68), polygenic risk scores, gene-expression signatures and
network inference (the graphical lasso of Module 18). This notebook derives ridge and lasso from first principles,
implements both from scratch, and tests how well they find causal SNPs when we know the answer.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import optimize, stats
from sklearn.linear_model import ElasticNet, Lasso, Ridge

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`load_prostate` gives the real data of slides 41–43; `simulate_genotypes` and `simulate_quantitative_trait` build cohorts with known causal SNPs; `standardize` z-scores columns (with training statistics); `precision_recall_f1` scores a set of selected SNPs against the truth. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import io
import numpy as np
import pandas as pd

PROSTATE_CSV = """lcavol,lweight,age,lbph,svi,lcp,gleason,pgg45,lpsa,train
-0.579818,2.76946,50,-1.38629,0,-1.38629,6,0,-0.430783,T
-0.994252,3.31963,58,-1.38629,0,-1.38629,6,0,-0.162519,T
-0.510826,2.69124,74,-1.38629,0,-1.38629,7,20,-0.162519,T
-1.20397,3.28279,58,-1.38629,0,-1.38629,6,0,-0.162519,T
0.751416,3.43237,62,-1.38629,0,-1.38629,6,0,0.371564,T
-1.04982,3.22883,50,-1.38629,0,-1.38629,6,0,0.765468,T
0.737164,3.47352,64,0.615186,0,-1.38629,6,0,0.765468,F
0.693147,3.53951,58,1.53687,0,-1.38629,6,0,0.854415,T
-0.776529,3.53951,47,-1.38629,0,-1.38629,6,0,1.04732,F
0.223144,3.24454,63,-1.38629,0,-1.38629,6,0,1.04732,F
0.254642,3.60414,65,-1.38629,0,-1.38629,6,0,1.26695,T
-1.34707,3.59868,63,1.26695,0,-1.38629,6,0,1.26695,T
1.61343,3.02286,63,-1.38629,0,-0.597837,7,30,1.26695,T
1.47705,2.99823,67,-1.38629,0,-1.38629,7,5,1.34807,T
1.20597,3.44202,57,-1.38629,0,-0.430783,7,5,1.39872,F
1.54116,3.06105,66,-1.38629,0,-1.38629,6,0,1.44692,T
-0.415515,3.51601,70,1.24415,0,-0.597837,7,30,1.47018,T
2.28849,3.64936,66,-1.38629,0,0.371564,6,0,1.4929,T
-0.562119,3.26767,41,-1.38629,0,-1.38629,6,0,1.55814,T
0.182322,3.82538,70,1.65823,0,-1.38629,6,0,1.59939,T
1.1474,3.41936,59,-1.38629,0,-1.38629,6,0,1.639,T
2.05924,3.50104,60,1.47476,0,1.34807,7,20,1.65823,F
-0.544727,3.37588,59,-0.798508,0,-1.38629,6,0,1.69562,T
1.78171,3.45157,63,0.438255,0,1.17866,7,60,1.7138,T
0.385262,3.6674,69,1.59939,0,-1.38629,6,0,1.73166,F
1.44692,3.12457,68,0.300105,0,-1.38629,6,0,1.76644,F
0.512824,3.71965,65,-1.38629,0,-0.798508,7,70,1.80006,T
-0.400478,3.86598,67,1.81645,0,-1.38629,7,20,1.81645,F
1.04028,3.12895,67,0.223144,0,0.0487902,7,80,1.84845,T
2.40964,3.37588,65,-1.38629,0,1.61939,6,0,1.89462,T
0.285179,4.09017,65,1.96291,0,-0.798508,6,0,1.92425,T
0.182322,3.80444,65,1.70475,0,-1.38629,6,0,2.00821,F
1.27536,3.03735,71,1.26695,0,-1.38629,6,0,2.00821,T
0.00995033,3.26767,54,-1.38629,0,-1.38629,6,0,2.02155,F
-0.0100503,3.21687,63,-1.38629,0,-0.798508,6,0,2.04769,T
1.30833,4.11985,64,2.17134,0,-1.38629,7,5,2.08567,F
1.42311,3.65713,73,-0.579819,0,1.65823,8,15,2.15756,T
0.457425,2.37491,64,-1.38629,0,-1.38629,7,15,2.19165,T
2.66096,4.08514,68,1.37372,1,1.83258,7,35,2.21375,T
0.797507,3.01308,56,0.936093,0,-0.162519,7,5,2.27727,T
0.620576,3.142,60,-1.38629,0,-1.38629,9,80,2.29757,T
1.4422,3.68261,68,-1.38629,0,-1.38629,7,10,2.30757,F
0.582216,3.86598,62,1.7138,0,-0.430783,6,0,2.32728,T
1.77156,3.89691,61,-1.38629,0,0.81093,7,6,2.37491,F
1.48614,3.4095,66,1.7492,0,-0.430783,7,20,2.52172,T
1.66393,3.39283,61,0.615186,0,-1.38629,7,15,2.55334,T
2.72785,3.99545,79,1.87947,1,2.65676,9,100,2.56879,T
1.16315,4.03512,68,1.7138,0,-0.430783,7,40,2.56879,F
1.74572,3.49802,43,-1.38629,0,-1.38629,6,0,2.59152,F
1.22083,3.56812,70,1.37372,0,-0.798508,6,0,2.59152,F
1.09192,3.9936,68,-1.38629,0,-1.38629,7,50,2.65676,T
1.66013,4.23483,64,2.07317,0,-1.38629,6,0,2.67759,T
0.512824,3.63363,64,1.4929,0,0.0487902,7,70,2.68444,F
2.12704,4.12147,68,1.76644,0,1.44692,7,40,2.69124,F
3.15359,3.51601,59,-1.38629,0,-1.38629,7,5,2.70471,F
1.26695,4.28013,66,2.12226,0,-1.38629,7,15,2.718,T
0.97456,2.86505,47,-1.38629,0,0.500775,7,4,2.78809,F
0.463734,3.76468,49,1.42311,0,-1.38629,6,0,2.79423,T
0.542324,4.17823,70,0.438255,0,-1.38629,7,20,2.80639,T
1.06126,3.85121,61,1.29473,0,-1.38629,7,40,2.81241,T
0.457425,4.5245,73,2.3263,0,-1.38629,6,0,2.842,T
1.99742,3.71965,63,1.61939,1,1.90954,7,40,2.85359,F
2.77571,3.52489,72,-1.38629,0,1.55814,9,95,2.85359,T
2.03471,3.91701,66,2.00821,1,2.11021,7,60,2.882,F
2.07317,3.62301,64,-1.38629,0,-1.38629,6,0,2.882,F
1.45862,3.83622,61,1.32176,0,-0.430783,7,20,2.88759,F
2.02287,3.87847,68,1.78339,0,1.32176,7,70,2.92047,T
2.19834,4.05091,72,2.30757,0,-0.430783,7,10,2.96269,T
-0.446287,4.40855,69,-1.38629,0,-1.38629,6,0,2.96269,T
1.19392,4.78038,72,2.3263,0,-0.798508,7,5,2.97298,T
1.86408,3.59319,60,-1.38629,1,1.32176,7,60,3.01308,T
1.16002,3.34109,77,1.7492,0,-1.38629,7,25,3.03735,T
1.21491,3.82538,69,-1.38629,1,0.223144,7,20,3.05636,F
1.83896,3.23672,60,0.438255,1,1.17866,9,90,3.07501,F
2.99923,3.84908,69,-1.38629,1,1.90954,7,20,3.27526,T
3.14113,3.26385,68,-0.0512933,1,2.42037,7,50,3.33755,T
2.01089,4.43379,72,2.12226,0,0.500775,7,60,3.39283,T
2.53766,4.35478,78,2.3263,0,-1.38629,7,10,3.4356,T
2.6483,3.58213,69,-1.38629,1,2.584,7,70,3.45789,T
2.77944,3.82319,63,-1.38629,0,0.371564,7,50,3.51304,F
1.46787,3.07038,66,0.559616,0,0.223144,7,40,3.51601,T
2.51366,3.47352,57,0.438255,0,2.32728,7,60,3.53076,T
2.61301,3.88875,77,-0.527633,1,0.559616,7,30,3.5653,T
2.67759,3.83838,65,1.11514,0,1.7492,9,70,3.57094,F
1.56235,3.70991,60,1.69562,0,0.81093,7,30,3.58768,T
3.30285,3.51898,64,-1.38629,1,2.32728,7,60,3.63099,T
2.02419,3.7317,58,1.639,0,-1.38629,6,0,3.68009,T
1.73166,3.36902,62,-1.38629,1,0.300105,7,30,3.71235,T
2.80759,4.71805,65,-1.38629,1,2.46385,7,60,3.98434,T
1.56235,3.69511,76,0.936093,1,0.81093,7,75,3.9936,T
3.24649,4.10182,68,-1.38629,0,-1.38629,6,0,4.02981,T
2.5329,3.67757,61,1.34807,1,-1.38629,7,15,4.12955,T
2.83027,3.8764,68,-1.38629,1,1.32176,7,60,4.38515,T
3.821,3.89691,44,-1.38629,1,2.16905,7,40,4.68444,T
2.90745,3.39619,52,-1.38629,1,2.46385,7,10,5.14312,F
2.88256,3.77391,68,1.55814,1,1.55814,7,80,5.47751,T
3.47197,3.975,68,0.438255,1,2.90417,7,20,5.58293,F
"""


def load_prostate() -> pd.DataFrame:
    """Prostate cancer data of 02-620 Lecture 2 (97 men, 8 clinical predictors, outcome ``lpsa``)."""
    df = pd.read_csv(io.StringIO(PROSTATE_CSV))
    df["train"] = df["train"] == "T"
    return df


def simulate_genotypes(rng: np.random.Generator, n_samples: int, n_snps: int, n_pops: int = 1,
                       fst: float = 0.0, maf_range=(0.05, 0.5)):
    """Simulate 0/1/2 genotypes under Hardy–Weinberg equilibrium.

    Each SNP gets an ancestral allele frequency drawn uniformly from ``maf_range``.  With
    ``n_pops > 1`` and ``fst > 0`` every population's frequency is drawn from the
    Balding–Nichols model Beta(p(1−F)/F, (1−p)(1−F)/F).  Samples are split evenly across
    populations.  Returns ``(G, pop, freqs)``: the N × J genotype matrix, the population of
    each sample and the n_pops × J matrix of population allele frequencies.

    ``G`` is stored as int8 to save memory: convert with ``G.astype(float)`` before arithmetic
    whose intermediate results can exceed 127.  A rare SNP can come out monomorphic (no
    variation) in a small sample — skip such columns in per-SNP tests.
    """
    p = rng.uniform(*maf_range, n_snps)
    if n_pops > 1 and fst > 0:
        a, b = p * (1 - fst) / fst, (1 - p) * (1 - fst) / fst
        freqs = np.clip(rng.beta(a, b, size=(n_pops, n_snps)), 0.001, 0.999)
    else:
        freqs = np.tile(p, (n_pops, 1))
    pop = np.repeat(np.arange(n_pops), int(np.ceil(n_samples / n_pops)))[:n_samples]
    G = rng.binomial(2, freqs[pop]).astype(np.int8)
    return G, pop, freqs


def simulate_quantitative_trait(rng: np.random.Generator, G: np.ndarray, n_causal: int = 5,
                                h2: float = 0.5, causal=None):
    """Simulate a quantitative phenotype y = G·β + ε with ``n_causal`` causal SNPs.

    The genetic part is scaled so that it explains a fraction ``h2`` (the heritability) of
    the phenotypic variance.  Returns ``(y, beta, causal_idx)``.
    """
    N, J = G.shape
    causal_idx = np.sort(rng.choice(J, n_causal, replace=False) if causal is None else np.asarray(causal))
    beta = np.zeros(J)
    beta[causal_idx] = rng.choice([-1, 1], len(causal_idx)) * rng.uniform(0.5, 1.0, len(causal_idx))
    g = G @ beta
    scale = np.sqrt(h2 / g.var()) if g.var() > 0 else 0.0
    beta *= scale
    y = G @ beta + rng.normal(0, np.sqrt(1 - h2), N)
    return y, beta, causal_idx


def standardize(X, mean=None, sd=None):
    """Z-score the columns of X. Pass the training mean/sd to transform test data the same way.

    Returns ``(Z, mean, sd)``; constant columns get sd = 1 so they map to 0.
    """
    X = np.asarray(X, float)
    mean = X.mean(axis=0) if mean is None else mean
    sd = X.std(axis=0) if sd is None else sd
    sd = np.where(sd > 0, sd, 1.0)
    return (X - mean) / sd, mean, sd


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


print("🧰 Toolkit loaded:", ", ".join(['load_prostate', 'simulate_genotypes', 'simulate_quantitative_trait', 'standardize', 'confusion_counts', 'precision_recall_f1']))

---
## 1 · When least squares fails — and what a prior can do

> 📍 **Where we are.** Module 06 derived $\mathbf w_{MLE} = (\mathbf X^\top\mathbf X)^{-1}\mathbf X^\top\mathbf y$ (slide 45 recaps
> it) and showed that it breaks down when the number of samples $N$ is smaller than the number of inputs $J$ (slide 46:
> "consider 3 million SNPs and 1,000 individuals"). This section recaps the failure on a small cohort that we will reuse,
> then sets up the Bayesian fix of slides 47–48.

⚠️ Slide 45 writes the model as $\mathbf y = \mathbf X^\top\mathbf w + \boldsymbol\epsilon$; with $\mathbf X$ stored as $N$ rows
(people) × $J$ columns (SNPs), as everywhere in this course, it should read $\mathbf y = \mathbf X\mathbf w + \boldsymbol\epsilon$.

In [ ]:
# ── 1.1 Recap: 50 people, 200 SNPs — the normal equations have no unique solution ──
rng = np.random.default_rng(SEED + 1)
G1, _, _ = simulate_genotypes(rng, 1050, 200)                    # 50 people to learn from + 1,000 new people
y1, w1_true, causal1 = simulate_quantitative_trait(rng, G1, n_causal=5, h2=0.5)
train1, test1 = np.arange(50), np.arange(50, 1050)
N1, J1 = len(train1), G1.shape[1]

mu1 = G1[train1].mean(axis=0)
X1 = G1[train1] - mu1                       # centred genotypes: Section 4 shows centring removes the intercept
y1c = y1[train1] - y1[train1].mean()
eig_ols = np.linalg.eigvalsh(X1.T @ X1)    # eigenvalues of the 200 × 200 matrix XᵀX
print(f"N = {N1} people, J = {J1} SNPs; rank of X = {np.linalg.matrix_rank(X1)}")
print(f"zero eigenvalues of XᵀX: {np.sum(eig_ols < 1e-8 * eig_ols.max())} of {J1}  →  XᵀX is singular")
w_minnorm = np.linalg.lstsq(X1, y1c, rcond=None)[0]               # one of infinitely many perfect fits
pred = y1[train1].mean() + (G1[test1] - mu1) @ w_minnorm
print(f"a perfect least-squares fit: training MSE {np.mean((y1c - X1 @ w_minnorm) ** 2):.1e}, "
      f"test MSE on 1,000 new people {np.mean((y1[test1] - pred) ** 2):.2f} "
      f"(predicting the mean: {np.mean((y1[test1] - y1[train1].mean()) ** 2):.2f})")

🔍 **What just happened?** The centred $50\times200$ genotype matrix has rank 49 — centring uses up one dimension — so
$\mathbf X^\top\mathbf X$ has 151 zero eigenvalues and no inverse. A perfect fit to the 50 training people exists, but it
predicts new people barely better than simply guessing the average phenotype (test MSE 0.97 vs 1.01). The data alone do not pin
down $\mathbf w$.

### 1.1 Bayes' rule for the weights (slides 47–48)

Module 05 offered two ways to estimate parameters: MLE (data only) and **MAP** (data *and* prior beliefs). In the Bayesian
view, the weights $\mathbf w$ are random variables with a distribution. Slide 47 derives the posterior with the product rule
$P(a,b) = P(a\mid b)P(b)$, applied to the joint distribution of $\mathbf w$, $y$ and $\mathbf x$ in two different orders:

$$p(\mathbf w, y, \mathbf x) = p(\mathbf w\mid y,\mathbf x)\,p(y,\mathbf x) = p(y\mid\mathbf x,\mathbf w)\,p(\mathbf x,\mathbf w).$$

*In words:* the joint probability can be split as "posterior × probability of the data" or as "likelihood × probability of
the inputs and weights" — both are correct, so they are equal.

The genotypes do not depend on the regression weights, so $p(\mathbf x,\mathbf w) = p(\mathbf x)\,p(\mathbf w)$. Dividing by
$p(y,\mathbf x)$, which does not depend on $\mathbf w$:

$$p(\mathbf w\mid y,\mathbf x) = \frac{p(y\mid\mathbf x,\mathbf w)\,p(\mathbf x)\,p(\mathbf w)}{p(y,\mathbf x)}
\;\propto\; \underbrace{p(y\mid\mathbf x,\mathbf w)}_{\text{likelihood}}\;\underbrace{p(\mathbf w)}_{\text{prior}} .$$

*In words:* **posterior ∝ likelihood × prior** — after seeing the data, a weight vector is plausible if it explains the
phenotypes well *and* was plausible beforehand. The symbol $\propto$ ("proportional to") means "equal up to a factor that
does not depend on $\mathbf w$", which cannot change where the maximum is.

The likelihood is the regression model of Module 06, $p(y\mid\mathbf x,\mathbf w) = \mathcal N(\mathbf x^\top\mathbf w,\sigma^2)$
(slide 48). The **MAP estimate** is the most probable weight vector a posteriori:
$\mathbf w_{MAP} = \arg\max_{\mathbf w}\, p(\mathbf w\mid\mathbf y,\mathbf X)$. All that remains is to choose the prior.

---
## 2 · MAP with a Gaussian prior is ridge regression

> 📍 **Where we are.** We have posterior ∝ likelihood × prior. Slide 49 chooses the prior; slides 50–54 turn the MAP
> problem into a formula.

**The prior (slide 49).** Each SNP's weight is a real number, and before seeing data we have no reason to expect positive
rather than negative effects, but we do believe effects are *small*. A natural choice is a Gaussian centred at zero,
independently for every SNP:

$$p(w_j) = \mathcal N(0,\sigma_0^2)\quad\text{for } j = 1,\dots,J, \qquad\text{equivalently}\qquad
p(\mathbf w) = \mathcal N(\mathbf 0,\ \sigma_0^2\mathbf I).$$

*In words:* every weight is believed to lie near zero, within about $\pm2\sigma_0$; the prior "pulls the regression
coefficients toward 0", and a **lower $\sigma_0$ means a stronger pull** (slide 49).

**No prior on the intercept $w_0$ (slide 49).** The intercept only sets the overall level of the phenotype (a mean blood
pressure of 120 mmHg, say); believing it to be near zero would make no sense. Section 4 shows how to handle it. Until then
we work with **centred** data ($\mathbf y$ and every column of $\mathbf X$ have mean zero), for which no intercept is needed.

🧮 **Math refresher — the multivariate Gaussian with covariance $\sigma_0^2\mathbf I$.** $\mathbf I$ is the identity matrix
(1 on the diagonal, 0 elsewhere), so $\sigma_0^2\mathbf I$ is the diagonal matrix drawn on slide 49: every weight has variance
$\sigma_0^2$ and every pair of weights has covariance 0 (independent). For independent variables the joint density is the
product of the individual densities:

$$p(\mathbf w) = \prod_{j=1}^{J}\frac{1}{\sqrt{2\pi\sigma_0^2}}\exp\!\Big(-\frac{w_j^2}{2\sigma_0^2}\Big)
= \frac{1}{(2\pi\sigma_0^2)^{J/2}}\exp\!\Big(-\frac{1}{2\sigma_0^2}\,\mathbf w^\top\mathbf w\Big).$$

*In words:* multiplying $J$ bell curves multiplies their constants (giving the power $J/2$) and adds their exponents
(giving $\sum_j w_j^2 = \mathbf w^\top\mathbf w$). The density depends on $\mathbf w$ only through its squared length, so its
contours are circles (spheres) around the origin.

⚠️ **Typo on slides 50–51.** The prior density there is written as $\mathcal N(0,\sigma^2\mathbf I) = \frac{1}{(2\pi)^{1/2}}
\frac{1}{\det(\sigma^2\mathbf I)^{1/2}}\exp(-\frac{1}{2\sigma^2}\mathbf w^\top\mathbf w)$. It should use the *prior* variance
$\sigma_0^2$ (not the noise variance $\sigma^2$) and $(2\pi)^{J/2}$; note $\det(\sigma_0^2\mathbf I) = (\sigma_0^2)^J$, so the
two forms of the constant agree. The constants do not affect the arg max, and the final MAP objective on slide 50 is correct.

In [ ]:
# ── 2.1 What the Gaussian prior believes ──
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
grid_w = np.linspace(-2, 2, 400)
for s0, col in zip([0.2, 0.5, 1.0], ["tab:red", "tab:orange", "tab:blue"]):
    axes[0].plot(grid_w, stats.norm.pdf(grid_w, 0, s0), color=col, label=f"σ0 = {s0}  (λ = {0.5 / s0 ** 2:g} if σ² = 0.5)")
axes[0].set_xlabel("weight w_j of one SNP"); axes[0].set_ylabel("prior density p(w_j)")
axes[0].set_title("smaller σ0 = stronger belief that effects are small"); axes[0].legend(fontsize=7, loc="upper left")
W1, W2 = np.meshgrid(grid_w, grid_w)
logp = stats.norm.logpdf(W1, 0, 0.5) + stats.norm.logpdf(W2, 0, 0.5)   # independent ⇒ log densities add
axes[1].contour(W1, W2, np.exp(logp), levels=6, cmap="Oranges")
axes[1].set_aspect("equal"); axes[1].set_xlabel("w_1"); axes[1].set_ylabel("w_2")
axes[1].set_title("prior N(0, σ0²I) for two SNPs (σ0 = 0.5): circles")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In one dimension, a small $\sigma_0$ concentrates the prior near zero; in two dimensions the prior
contours are circles, because only the length of $\mathbf w$ matters. Now combine this prior with the likelihood.

### 2.1 The MAP objective (slides 50–51)

Take logs (the arg max does not move; products become sums) and plug in both Gaussians — the likelihood from Module 06,
the prior from above:

$$\begin{aligned}
\mathbf w_{MAP} &= \underset{\mathbf w}{\arg\max}\ \big[\log p(\mathbf y\mid\mathbf X,\mathbf w) + \log p(\mathbf w)\big] &(1)\\
&= \underset{\mathbf w}{\arg\max}\ \Big[-\tfrac N2\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w)
   \;-\; \tfrac J2\log(2\pi\sigma_0^2) - \frac{1}{2\sigma_0^2}\mathbf w^\top\mathbf w\Big] &(2)\\
&= \underset{\mathbf w}{\arg\min}\ \Big[\frac{1}{2\sigma^2}(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \frac{1}{2\sigma_0^2}\mathbf w^\top\mathbf w\Big] &(3)\\
&= \underset{\mathbf w}{\arg\min}\ \Big[(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \frac{\sigma^2}{\sigma_0^2}\,\mathbf w^\top\mathbf w\Big] &(4)
\end{aligned}$$

*In words, line by line:* (1) maximise log likelihood + log prior; (2) the log of each Gaussian is "log of its constant"
minus "the quadratic in its exponent"; (3) drop the two log-constant terms (they do not contain $\mathbf w$) and flip the
sign, turning arg max into arg min; (4) multiply everything by the positive constant $2\sigma^2$.

*In words, the result:* **the MAP estimate minimises the squared error plus a penalty on the squared length of
$\mathbf w$**, weighted by the ratio of noise variance to prior variance.

### 2.2 Solving it (slides 52–53)

Differentiate (4) and set the gradient to zero. The first term's gradient is $-2\mathbf X^\top\mathbf y + 2\mathbf X^\top\mathbf X\mathbf w$
(Module 06). For the second, use $\frac{\partial}{\partial\mathbf w}\mathbf w^\top\mathbf A\mathbf w = (\mathbf A+\mathbf A^\top)\mathbf w$
with $\mathbf A = \mathbf I$: $\frac{\partial}{\partial\mathbf w}\mathbf w^\top\mathbf w = 2\mathbf w$. So

$$-2\mathbf X^\top\mathbf y + 2\mathbf X^\top\mathbf X\mathbf w + \frac{2\sigma^2}{\sigma_0^2}\mathbf w = \mathbf 0 .$$

*In words:* at the optimum, the pull of the data (first two terms) balances the pull of the prior toward zero (last term).

Divide by 2 and write $\mathbf w = \mathbf I\mathbf w$ (inserting the identity, slide 52's trick) so that the two terms in
$\mathbf w$ can be added:

$$\Big(\mathbf X^\top\mathbf X + \frac{\sigma^2}{\sigma_0^2}\mathbf I\Big)\mathbf w = \mathbf X^\top\mathbf y
\quad\Longrightarrow\quad
\boxed{\ \mathbf w_{MAP} = \Big(\mathbf X^\top\mathbf X + \frac{\sigma^2}{\sigma_0^2}\mathbf I\Big)^{-1}\mathbf X^\top\mathbf y\ }$$

*In words:* the MAP estimate is the least-squares formula with the number $\sigma^2/\sigma_0^2$ added to every diagonal
entry of $\mathbf X^\top\mathbf X$ — a "ridge" along the diagonal, which gave **ridge regression** its name (slide 54's notes).

Let us verify all of this numerically on the 50-person cohort: write the negative log posterior directly with Gaussian
log-densities (no algebra), minimise it with a general-purpose optimiser, and compare with the boxed formula.

In [ ]:
# ── 2.2 Check: the boxed formula is the maximum of the posterior ──
sigma2, sigma0_2 = 0.5, 0.05                     # noise variance (h² = 0.5 ⇒ 0.5) and a fairly strong prior
lam1 = sigma2 / sigma0_2                         # = 10


def ridge_weights(Xc, yc, lam):
    """MAP / ridge weights (XᵀX + λI)⁻¹Xᵀy for centred data, by solving the linear system."""
    return np.linalg.solve(Xc.T @ Xc + lam * np.eye(Xc.shape[1]), Xc.T @ yc)


def neg_log_posterior(w, X, y, sigma2, sigma0_2):
    """−[log p(y | X, w) + log p(w)], written directly with Gaussian log-densities (no algebra used)."""
    log_lik = np.sum(stats.norm.logpdf(y, loc=X @ w, scale=np.sqrt(sigma2)))
    log_prior = np.sum(stats.norm.logpdf(w, loc=0.0, scale=np.sqrt(sigma0_2)))
    return -(log_lik + log_prior)


def neg_log_posterior_grad(w, X, y, sigma2, sigma0_2):
    """Gradient of the negative log posterior from our derivation: (−Xᵀy + XᵀXw)/σ² + w/σ0²."""
    return (-X.T @ y + X.T @ X @ w) / sigma2 + w / sigma0_2


w_map = ridge_weights(X1, y1c, lam1)
w_probe = rng.normal(0, 0.1, J1)                 # check the gradient formula by finite differences
eps, idx = 1e-6, rng.choice(J1, 5, replace=False)
fd = [(neg_log_posterior(w_probe + eps * np.eye(J1)[j], X1, y1c, sigma2, sigma0_2)
       - neg_log_posterior(w_probe - eps * np.eye(J1)[j], X1, y1c, sigma2, sigma0_2)) / (2 * eps) for j in idx]
print("finite differences:", np.round(fd, 4))
print("gradient formula  :", np.round(neg_log_posterior_grad(w_probe, X1, y1c, sigma2, sigma0_2)[idx], 4))
res = optimize.minimize(neg_log_posterior, np.zeros(J1), args=(X1, y1c, sigma2, sigma0_2),
                        jac=neg_log_posterior_grad, method="L-BFGS-B", options=dict(maxiter=10000, gtol=1e-10))
print(f"optimiser converged: {res.success}; max |w_optimiser − w_formula| = {np.abs(res.x - w_map).max():.2e}")
print(f"gradient at the formula's answer (should be 0): {np.abs(neg_log_posterior_grad(w_map, X1, y1c, sigma2, sigma0_2)).max():.2e}")

🔍 **What just happened?** The gradient we derived matches brute-force finite differences of the log posterior, the gradient
vanishes at the boxed formula's answer, and a general optimiser that knows nothing about our algebra lands on the same
200 weights (to about six decimals — the optimiser's own tolerance). The derivation is right — and it worked with $N = 50 <
J = 200$, where least squares had no unique answer.

To *see* what the prior does, shrink the problem to two SNPs in linkage disequilibrium and 20 people, and draw the
likelihood, the prior and the posterior as contour maps over $(w_1, w_2)$.

In [ ]:
# ── 2.3 Likelihood × prior = posterior, for two correlated SNPs ──
rng2 = np.random.default_rng(SEED + 2)
n2 = 20
xa = rng2.binomial(2, 0.4, n2)
xb = np.where(rng2.random(n2) < 0.7, xa, rng2.binomial(2, 0.4, n2))   # SNP b partly copies SNP a (LD)
X2 = np.column_stack([xa, xb]).astype(float); X2 -= X2.mean(axis=0)
y2 = 0.6 * X2[:, 0] + rng2.normal(0, 1, n2); y2 -= y2.mean()
s2_noise, s0_prior = 1.0, 0.3
w_mle2 = np.linalg.solve(X2.T @ X2, X2.T @ y2)
w_map2 = ridge_weights(X2, y2, s2_noise / s0_prior ** 2)

g = np.linspace(-1.2, 1.8, 241)
A1, A2 = np.meshgrid(g, g)
Wg = np.stack([A1.ravel(), A2.ravel()], axis=1)                      # every grid point is a candidate w
loglik = (-0.5 / s2_noise * np.sum((y2[None, :] - Wg @ X2.T) ** 2, axis=1)).reshape(A1.shape)
logprior = (-0.5 / s0_prior ** 2 * np.sum(Wg ** 2, axis=1)).reshape(A1.shape)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6), sharey=True)
for ax, Z, title, cmap in zip(axes, [loglik, logprior, loglik + logprior],
                              ["likelihood p(y | X, w)", f"prior N(0, {s0_prior}²I)", "posterior ∝ likelihood × prior"],
                              ["Blues", "Oranges", "Greens"]):
    ax.contour(A1, A2, np.exp(Z - Z.max()), levels=[0.05, 0.2, 0.4, 0.6, 0.8, 0.95], cmap=cmap)
    ax.plot(*w_mle2, "b^", ms=8, label="MLE (least squares)"); ax.plot(*w_map2, "g*", ms=12, label="MAP (ridge)")
    ax.plot(0, 0, "o", color="tab:orange", ms=6, label="prior mean 0")
    ax.set_title(title); ax.set_xlabel("w_1 (SNP a, causal)"); ax.set_aspect("equal")
path = np.array([ridge_weights(X2, y2, s2_noise / s0 ** 2) for s0 in np.logspace(1, -1.5, 60)])
axes[2].plot(path[:, 0], path[:, 1], "k:", lw=1, label="MAP as σ0 shrinks")
axes[0].set_ylabel("w_2 (SNP b)"); axes[2].legend(fontsize=6, loc="upper right")
plt.tight_layout(); plt.show()
print("MLE:", np.round(w_mle2, 3), "  MAP:", np.round(w_map2, 3), "  true: [0.6, 0]")

🔍 **What just happened?** With only 20 people and two correlated SNPs, the likelihood (blue) is a long, tilted ellipse:
the data say "$w_1 + w_2$ is about such-and-such" but cannot tell the two SNPs apart. The prior (orange) is a set of
circles around zero. Their product, the posterior (green), sits **between** them, and its peak — the MAP estimate — is the
least-squares estimate pulled toward the origin. The dotted line traces the MAP estimate as $\sigma_0$ decreases: from the
MLE (weak prior) all the way to $\mathbf 0$ (overwhelming prior).

### 2.3 Why ridge always has a solution (slide 54: "invertible, even if $N<J$")

From Module 03: if $\mathbf A\mathbf v = d\,\mathbf v$ (an eigenvector $\mathbf v$ with eigenvalue $d$), then
$(\mathbf A + \lambda\mathbf I)\mathbf v = (d + \lambda)\mathbf v$. Adding $\lambda\mathbf I$ **raises every eigenvalue by
$\lambda$** and keeps the eigenvectors. The eigenvalues of $\mathbf X^\top\mathbf X$ are never negative (they are sums of
squares), so with $\lambda > 0$ every eigenvalue of $\mathbf X^\top\mathbf X + \lambda\mathbf I$ is at least $\lambda$ — none is
zero, and the matrix is invertible, whatever $N$ and $J$ are.

In [ ]:
# ── 2.4 Eigenvalues before and after adding λI, and what ridge buys on new people ──
eig_ridge = np.linalg.eigvalsh(X1.T @ X1 + lam1 * np.eye(J1))
print(f"XᵀX      : smallest eigenvalue {eig_ols.min():+.1e}, largest {eig_ols.max():.1f}")
print(f"XᵀX + λI : smallest eigenvalue {eig_ridge.min():+.3f}, largest {eig_ridge.max():.1f}  (λ = {lam1:g})")
print("eigenvalues shifted by exactly λ:", np.allclose(eig_ridge, eig_ols + lam1))
pred_ridge = y1[train1].mean() + (G1[test1] - mu1) @ w_map
print(f"test MSE on 1,000 new people: ridge {np.mean((y1[test1] - pred_ridge) ** 2):.2f} | "
      f"perfect least-squares fit {np.mean((y1[test1] - pred) ** 2):.2f} | "
      f"predict the mean {np.mean((y1[test1] - y1[train1].mean()) ** 2):.2f}")

🔍 **What just happened?** The 151 zero eigenvalues became exactly $\lambda = 10$; the matrix is now comfortably invertible.
On 1,000 new people, ridge predicts better than both the perfect-fit least-squares solution and the mean. The improvement
is modest — 50 people cannot reveal much about 200 SNPs — but the method no longer breaks.

✏️ **Your turn.** In cell 2.2 change `sigma0_2` to `0.005` (stronger prior) and to `5.0` (weaker prior) and re-run cells
2.2 and 2.4. How do the size of the weights (`np.abs(w_map).max()`) and the test MSE change? Which extreme behaves like
"predict the mean", and which like least squares?

### 📑 Slide connection — `002_regression`, slides 45–54

- **What the slides show.** The limitation of least squares when $N<J$ (slide 46), Bayes' rule $p(\mathbf w\mid y,\mathbf x)\propto p(y\mid\mathbf x,\mathbf w)\,p(\mathbf w)$ (slides 47–48), the Gaussian prior $p(w_i) = \mathcal N(0,\sigma_0^2)$ with no prior on $w_0$ ("pulls the coefficients toward 0; lower $\sigma_0$ means stronger pulling"), the MAP objective $\arg\min(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \frac{\sigma^2}{\sigma_0^2}\mathbf w^\top\mathbf w$, its derivative and the solution $(\mathbf X^\top\mathbf X + \frac{\sigma^2}{\sigma_0^2}\mathbf I)^{-1}\mathbf X^\top\mathbf y$, "invertible, even if $N<J$".
- **What this notebook adds.** Where $p(\mathbf x)$ goes in Bayes' rule, the product form of the isotropic Gaussian, every algebra step with a reason, a check by brute-force optimisation of the log posterior, the eigenvalue argument for invertibility with numbers, and the likelihood/prior/posterior picture. We also flag two slide typos: $\mathbf y = \mathbf X^\top\mathbf w$ on slide 45 (should be $\mathbf X\mathbf w$) and the prior density on slides 50–51 (should use $\sigma_0^2$ and $(2\pi)^{J/2}$).
- **Exam tip.** Be able to go from "Gaussian likelihood + Gaussian prior" to $(\mathbf X^\top\mathbf X + \lambda\mathbf I)^{-1}\mathbf X^\top\mathbf y$ in five lines, and to say what $\lambda$ is in terms of $\sigma^2$ and $\sigma_0^2$. Common error: writing $\lambda = \sigma_0^2/\sigma^2$ (upside down) — check with the limit $\sigma_0\to\infty$, which must give back least squares.

---
## 3 · Ridge regression as regularisation (slides 55–57)

> 📍 **Where we are.** We derived ridge as a MAP estimate. Slides 55–57 give the frequentist reading of the same formula
> and the vocabulary used in every ML paper.

Write $\lambda = \sigma^2/\sigma_0^2$. The MAP problem becomes

$$\mathbf w_{ridge} = \underset{\mathbf w}{\arg\min}\ (\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \lambda\,\mathbf w^\top\mathbf w
\;=\; \underset{\mathbf w}{\arg\min}\ \lVert\mathbf y-\mathbf X\mathbf w\rVert_2^2 + \lambda\lVert\mathbf w\rVert_2^2 .$$

*In words:* fit the data well, but pay a price $\lambda$ for every unit of squared weight. A frequentist calls the second
term a **regulariser** (or **penalty**) and $\lambda$ the **regularisation parameter**; a Bayesian calls it the log prior.
Same formula, two stories (slide 56).

🧮 **Math refresher — norms.** The **$L_2$ norm** is the ordinary length, $\lVert\mathbf w\rVert_2 = \sqrt{\sum_j w_j^2}$, so
$\lVert\mathbf w\rVert_2^2 = \mathbf w^\top\mathbf w$ (slide 57's "equivalently"). The **$L_1$ norm** is the sum of absolute values,
$\lVert\mathbf w\rVert_1 = \sum_j |w_j|$ — we need it for the lasso in Section 6.

**"Small $\sigma_0^2$ means large $\lambda$, which means stronger pull of the estimates toward zero" (slide 57).** Two limits:
$\lambda\to0$ (a flat prior, $\sigma_0\to\infty$) gives back least squares; $\lambda\to\infty$ (a prior certain that all effects
are zero) gives $\mathbf w = \mathbf 0$.

**The cleanest case: uncorrelated features.** If the columns of $\mathbf X$ are orthonormal ($\mathbf X^\top\mathbf X = \mathbf I$),
least squares gives $\mathbf w_{OLS} = \mathbf X^\top\mathbf y$ and ridge gives

$$\mathbf w_{ridge} = (\mathbf I + \lambda\mathbf I)^{-1}\mathbf X^\top\mathbf y = \frac{1}{1+\lambda}\,\mathbf w_{OLS}.$$

*In words:* ridge shrinks every least-squares coefficient by the same factor $1/(1+\lambda)$ — towards zero, but never
exactly to zero. (With correlated features the shrinkage is uneven, as the 2-SNP picture showed.)

In [ ]:
# ── 3.1 Check the orthonormal case: ridge = OLS / (1 + λ) ──
rng3 = np.random.default_rng(SEED + 3)
Q, _ = np.linalg.qr(rng3.normal(size=(100, 5)))          # 100 × 5 matrix with orthonormal columns: QᵀQ = I
yq = Q @ np.array([3.0, -2.0, 1.0, 0.5, 0.0]) + rng3.normal(0, 0.3, 100)
w_ols_q = Q.T @ yq
for lam in [0.5, 1.0, 4.0]:
    print(f"λ = {lam}: ridge / OLS = {np.round(ridge_weights(Q, yq, lam) / w_ols_q, 4)}   1/(1+λ) = {1 / (1 + lam):.4f}")

In [ ]:
# ── 3.2 The ridge path on the real prostate data ──
prostate = load_prostate()
predictors = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
is_train = prostate["train"].to_numpy()
X_raw = prostate[predictors].to_numpy(float)
y_psa = prostate["lpsa"].to_numpy()
Z_tr, mu_tr, sd_tr = standardize(X_raw[is_train])         # standardise with the 67 training men only
Z_te, _, _ = standardize(X_raw[~is_train], mu_tr, sd_tr)
y_tr, y_te = y_psa[is_train], y_psa[~is_train]
Zc_tr = Z_tr - Z_tr.mean(axis=0)                          # (already ≈ 0 after standardising, but be explicit)
yc_tr = y_tr - y_tr.mean()

lams_r = np.logspace(-2, 4, 80)
ridge_path = np.array([ridge_weights(Zc_tr, yc_tr, lam) for lam in lams_r])
fig, ax = plt.subplots(figsize=(7, 3.6))
for j, name in enumerate(predictors):
    ax.plot(lams_r, ridge_path[:, j], lw=1.5, label=name)
ax.axhline(0, color="k", lw=0.6)
ax.set_xscale("log"); ax.set_xlabel("λ (log scale)  →  stronger prior / more shrinkage")
ax.set_ylabel("coefficient (per SD of predictor)"); ax.legend(fontsize=7, ncol=2)
ax.set_title("ridge path, prostate data (REAL, 67 training men)")
plt.tight_layout(); plt.show()
print("λ = 0.01  :", np.round(ridge_path[0], 3))
print("λ = 10,000:", np.round(ridge_path[-1], 3))

🔍 **What just happened?** For orthonormal features the ratio ridge/OLS is exactly $1/(1+\lambda)$ for every coefficient.
On the prostate data, at tiny $\lambda$ the coefficients are the least-squares values of Module 06 (with training-only
standardisation: `lcavol` 0.71, `lweight` 0.29, …); as $\lambda$ grows, all of them shrink smoothly toward zero, and at
$\lambda = 10^4$ they are nearly zero but **none is exactly zero**. Correlated predictors shrink unevenly: `lcp`'s negative
coefficient, which rested on its correlation with `lcavol` and `svi`, shrinks fast and even turns slightly *positive* at
moderate $\lambda$ (so do `age` and `gleason`). There is a neat reason: for very large $\lambda$,
$(\mathbf X^\top\mathbf X + \lambda\mathbf I)^{-1}\approx\mathbf I/\lambda$, so $\mathbf w\approx\mathbf X^\top\mathbf y/\lambda$ — each
coefficient becomes proportional to its predictor's covariance with log PSA, the one-at-a-time view of Module 06. All eight
predictors correlate positively with log PSA, so all curves end above zero. Ridge interpolates between the joint fit
($\lambda\to0$) and the marginal one ($\lambda\to\infty$).

### 3.1 Why shrink at all? Bias and variance

Shrinking pulls estimates away from the least-squares values, so they become **biased** (on average not equal to the
truth). In exchange they **vary less** from one data set to another. When data are scarce, that trade is worth it. We can
watch it happen by repeating an experiment 300 times: draw a fresh training set of 40 people (30 SNPs, 5 causal), fit
ridge for several $\lambda$, and look at how the estimates and the predictions spread.

In [ ]:
# ── 3.3 Bias and variance of ridge, by repeating the experiment 300 times ──
rng_bv = np.random.default_rng(SEED + 4)
Gp, _, _ = simulate_genotypes(rng_bv, 4000, 30)
w_bv = np.zeros(30); w_bv[:5] = [1.0, -0.8, 0.6, 0.6, -0.4]           # 5 causal SNPs, 25 null
X_pool, X_new = Gp[:2000].astype(float), Gp[2000:].astype(float)
f_new = X_new @ w_bv                                                  # the true mean phenotype of 2,000 new people
lams_bv, reps, sigma_bv = [0, 0.3, 1, 3, 10, 30, 100, 300], 300, 1.0
W_hat = {lam: np.zeros((reps, 30)) for lam in lams_bv}
P_hat = {lam: np.zeros((reps, len(X_new))) for lam in lams_bv}
for r in range(reps):
    rows = rng_bv.choice(2000, 40, replace=False)                     # a fresh study of 40 people
    Xr = X_pool[rows]; yr = Xr @ w_bv + rng_bv.normal(0, sigma_bv, 40)
    mur, ybar = Xr.mean(axis=0), yr.mean()
    for lam in lams_bv:
        w = ridge_weights(Xr - mur, yr - ybar, lam) if lam > 0 else np.linalg.lstsq(Xr - mur, yr - ybar, rcond=None)[0]
        W_hat[lam][r], P_hat[lam][r] = w, ybar + (X_new - mur) @ w
bias2 = np.array([np.mean((P_hat[l].mean(axis=0) - f_new) ** 2) for l in lams_bv])   # squared bias of predictions
var = np.array([np.mean(P_hat[l].var(axis=0)) for l in lams_bv])                      # variance of predictions
print(pd.DataFrame({"λ": lams_bv, "mean of ŵ_1 (true 1.0)": [W_hat[l][:, 0].mean() for l in lams_bv],
                    "SD of ŵ_1": [W_hat[l][:, 0].std() for l in lams_bv], "bias²": bias2, "variance": var,
                    "expected test MSE = bias² + var + σ²": bias2 + var + sigma_bv ** 2}).round(3).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for lam, col in zip([0, 3, 30], ["tab:gray", "tab:blue", "tab:red"]):
    axes[0].hist(W_hat[lam][:, 0], bins=40, range=(-2, 4), alpha=0.55, color=col, label=f"λ = {lam}")
axes[0].axvline(1.0, color="k", ls="--", label="true w_1 = 1.0")
axes[0].set_xlabel("estimate of w_1 in 300 repeated studies"); axes[0].set_ylabel("count"); axes[0].legend(fontsize=7)
axes[0].set_title("shrinkage: narrower but shifted toward 0")
lx = np.array([0.1] + lams_bv[1:])                                    # plot λ = 0 at 0.1 on the log axis
axes[1].plot(lx, bias2, "o-", label="bias²"); axes[1].plot(lx, var, "s-", label="variance")
axes[1].plot(lx, bias2 + var + 1, "k^-", label="expected test MSE")
axes[1].set_xscale("log"); axes[1].set_yscale("log"); axes[1].set_xlabel("λ (leftmost point: λ = 0, least squares)")
axes[1].set_ylabel("mean squared error"); axes[1].legend(fontsize=7); axes[1].set_title("the bias–variance trade-off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** With $\lambda = 0$ (least squares, 40 people, 30 SNPs) the estimate of $w_1$ is unbiased (mean
close to the true 1.0) but very variable from study to study (SD about 0.5). As $\lambda$ grows, the histogram narrows and slides toward
zero: **bias up, variance down**. Expected test error — squared bias + variance + the irreducible noise $\sigma^2 = 1$ —
is very high for least squares, falls to a minimum at an intermediate $\lambda$ (around $\lambda = 10$ here, about a third of the least-squares error),
and rises again as
over-shrinkage makes the model ignore real effects. Choosing $\lambda$ well is the whole game (Section 5; Module 23 studies
the decomposition properly).

✏️ **Your turn.** Change the study size from 40 to 400 people (`rng_bv.choice(2000, 400, …)` and the noise vector length)
and re-run. Where is the best $\lambda$ now, and how much does ridge beat least squares? Why does more data make the prior
matter less?

### 📑 Slide connection — `002_regression`, slides 55–57

- **What the slides show.** MAP estimation viewed as **regularisation** of MLE: $\arg\min(\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \lambda\mathbf w^\top\mathbf w$ with regularisation parameter $\lambda$ ("also known as ridge regression"), equivalently $\lVert\mathbf y-\mathbf X\mathbf w\rVert_2^2 + \lambda\lVert\mathbf w\rVert_2^2$; "small $\sigma_0^2$ means large $\lambda$, which means stronger pull of the estimates toward zero".
- **What this notebook adds.** The orthonormal-design result $\mathbf w_{ridge} = \mathbf w_{OLS}/(1+\lambda)$ checked in code, the ridge path on the real prostate data (as in ESL Fig. 3.8, which plots it against effective degrees of freedom), and a 300-replicate simulation that measures bias², variance and test error as $\lambda$ changes — the "$\lambda\uparrow$ bias $\uparrow$, $\lambda\downarrow$ variance $\uparrow$" message of the model-selection extra slides.
- **Exam tip.** Know both readings of $\lambda$ (prior precision ratio $\sigma^2/\sigma_0^2$; penalty weight) and the two limits ($\lambda\to0$: OLS; $\lambda\to\infty$: $\mathbf w\to\mathbf 0$). Ridge shrinks but does **not** select: no coefficient becomes exactly zero for finite $\lambda$.

---
## 4 · Ridge regression in practice (slides 58–59)

> 📍 **Where we are.** So far we used centred data and ignored the intercept. Slides 58–59 give the practical recipe that
> handles the intercept correctly — and we add two pitfalls that bite in real analyses.

**The recipe.** We do not put a prior on the intercept $w_0$. Instead:

1. **Mean-centre the data:** $y_c^i = y^i - \bar y$ and $x^i_{c,j} = x^i_j - \bar x_j$ for every feature $j$;
2. **fit without an intercept** on the centred data: $\mathbf w = (\mathbf X_c^\top\mathbf X_c + \lambda\mathbf I)^{-1}\mathbf X_c^\top\mathbf y_c$;
3. **recover the intercept:** $w_0 = \bar y - [\bar x_1,\dots,\bar x_J]\,\mathbf w$.

*In words:* centring moves the cloud of data points so that it is centred at the origin; a fitted plane through the
centred cloud needs no intercept; step 3 moves the plane back.

**Why this is exactly right.** Minimise $\sum_i (y^i - w_0 - \mathbf x^{i\top}\mathbf w)^2 + \lambda\mathbf w^\top\mathbf w$ — the penalty
does not involve $w_0$. Setting the derivative with respect to $w_0$ to zero gives, exactly as in Module 06,

$$w_0 = \bar y - \bar{\mathbf x}^\top\mathbf w .$$

*In words:* whatever the slopes, the best intercept makes the fitted plane pass through the point of means.

Substituting back, each residual becomes $y^i - \bar y - (\mathbf x^i - \bar{\mathbf x})^\top\mathbf w = y_c^i - \mathbf x_c^{i\top}\mathbf w$, so what
remains to minimise is the centred ridge problem of step 2. ⚠️ Slide 58's centred matrix $\mathbf X_c$ has index typos
(its entries read $x^1_1 - \text{mean}(x_J)$ and $x^N_1 - \text{mean}(x_J)$ in the last column); the entry in row $i$,
column $j$ is $x^i_j - \text{mean}(x_j)$.

In [ ]:
# ── 4.1 The full ridge recipe, checked two independent ways ──
def fit_ridge(X, y, lam):
    """Ridge regression with an unpenalised intercept (slides 58–59). Returns (w0, w)."""
    x_bar, y_bar = X.mean(axis=0), y.mean()
    w = ridge_weights(X - x_bar, y - y_bar, lam)      # steps 1–2: centre, fit without intercept
    w0 = y_bar - x_bar @ w                            # step 3: recover the intercept
    return w0, w


lam_demo = 5.0
w0_r, w_r = fit_ridge(Z_tr, y_tr, lam_demo)
# Check 1: augmented X with a column of ones, and a penalty matrix that leaves w0 alone
A = np.column_stack([np.ones(len(y_tr)), Z_tr])
P = np.diag([0.0] + [1.0] * Z_tr.shape[1])            # diag(0, 1, …, 1): no penalty on the intercept
w_aug = np.linalg.solve(A.T @ A + lam_demo * P, A.T @ y_tr)
# Check 2: scikit-learn's Ridge minimises ||y − w0 − Xw||² + α||w||² (intercept unpenalised)
sk = Ridge(alpha=lam_demo).fit(Z_tr, y_tr)
print("recipe (centre → fit → recover):", np.round(np.r_[w0_r, w_r], 4))
print("augmented with diag(0,1,…,1)   :", np.round(w_aug, 4))
print("scikit-learn Ridge             :", np.round(np.r_[sk.intercept_, sk.coef_], 4))

🔍 **What just happened?** Three different computations — the slides' recipe, a direct solve with an intercept column whose
penalty is switched off, and scikit-learn — give identical numbers. (Note that scikit-learn's `Ridge(alpha=λ)` uses the
same scaling as the slides, $\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert^2$; its `Lasso` does **not**,
as we will see.) Now, what goes wrong if we *do* penalise the intercept?

In [ ]:
# ── 4.2 ⚠️ What goes wrong if the intercept is penalised: a blood-pressure-like trait ──
rng4 = np.random.default_rng(SEED + 5)
G4, _, _ = simulate_genotypes(rng4, 200, 50)
g4, _, _ = simulate_quantitative_trait(rng4, G4, n_causal=5, h2=0.5)
bp = 120 + 10 * g4                                      # systolic blood pressure in mmHg (SIMULATED)


def ridge_penalising_intercept(X, y, lam):
    """The WRONG way: put the column of ones into X and penalise its weight like every other weight."""
    A = np.column_stack([np.ones(len(y)), X])
    w = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ y)
    return w[0], w[1:]


X4 = G4.astype(float)
for name, fit in [("correct (slides 58–59)", fit_ridge), ("intercept penalised", ridge_penalising_intercept)]:
    w0, w = fit(X4, bp, 100.0)
    w0_s, w_s = fit(X4, bp + 100, 100.0)                # same people, blood pressure measured with a +100 offset
    print(f"{name:24s}: w0 = {w0:6.1f} mmHg, mean prediction {np.mean(w0 + X4 @ w):6.1f} (data mean {bp.mean():.1f}); "
          f"largest change in a SNP weight after adding 100 to y: {np.abs(w_s - w).max():.3f}")

🔍 **What just happened?** Penalising the intercept drags it from about 123 mmHg (the correct value: the predicted
pressure of a person carrying zero copies of every counted allele) down to about 10 mmHg; the SNP weights then distort
themselves to compensate, the average prediction no longer matches the average blood pressure, and — worst of
all — the estimated **genetic effects change when we merely shift the units of $y$** (measuring from a different zero point).
With the correct recipe, shifting $y$ moves only $w_0$; the SNP weights are untouched. This is why slide 49 says "we do not
place a prior on $w_0$".

**Second pitfall: the scale of the features.** The penalty $\lambda\sum_j w_j^2$ treats all weights alike, but a weight's size
depends on its feature's units. Record `pgg45` (percentage of Gleason scores 4 or 5) as a fraction between 0 and 1
instead of a percentage, and its coefficient must become 100 times larger to describe the same effect — so ridge penalises
it $100^2 = 10{,}000$ times more. Least squares does not care (Module 06, Section 7); ridge does.

In [ ]:
# ── 4.3 ⚠️ Units matter for ridge (but not for least squares): pgg45 as a percentage vs a fraction ──
X_frac = X_raw.copy(); X_frac[:, 7] /= 100                         # column 7 = pgg45: 0–100 % → 0–1
for label, Xu in [("pgg45 in percent ", X_raw), ("pgg45 as fraction", X_frac)]:
    w0_ls, w_ls = fit_ridge(Xu[is_train], y_tr, 1e-9)              # λ ≈ 0: least squares
    w0_rg, w_rg = fit_ridge(Xu[is_train], y_tr, 20.0)
    effect_per_sd = w_rg[7] * Xu[is_train, 7].std()                # pgg45's effect per SD: comparable across units
    print(f"{label}: least-squares test MSE {np.mean((y_te - w0_ls - Xu[~is_train] @ w_ls) ** 2):.4f} | "
          f"ridge (λ = 20) test MSE {np.mean((y_te - w0_rg - Xu[~is_train] @ w_rg) ** 2):.4f}, "
          f"pgg45 effect per SD {effect_per_sd:+.3f}")
w0_z, w_z = fit_ridge(Z_tr, y_tr, 20.0)
print(f"standardised (training men's mean/SD): ridge test MSE {np.mean((y_te - w0_z - Z_te @ w_z) ** 2):.4f} — "
      "the same whatever units pgg45 was recorded in")

🔍 **What just happened?** Least squares gives exactly the same test error (0.521) whichever way `pgg45` is recorded.
Ridge does not: with the same $\lambda$, the two versions give different predictions (test MSE 0.546 vs 0.480) and a
`pgg45` effect of 0.26 vs 0.03 per SD — although both contain identical information. That the fraction version happens to
predict better here is an accident of units, not a property of the biology. The standard
remedy is to **standardise** every feature (z-scores, computed on the **training** data only and then applied to new data)
before ridge or lasso, so that the penalty treats them on an equal footing. For SNP genotypes coded 0/1/2 the units are
already comparable, and both conventions (raw counts or standardised genotypes) are used in practice; standardising gives
rare variants relatively larger per-allele effects.

✏️ **Your turn.** Repeat cell 4.3 with `lbph` (column 3) multiplied by 1,000. Which method's test MSE changes, and in which
direction does the `lbph` coefficient move under ridge?

### 📑 Slide connection — `002_regression`, slides 58–59

- **What the slides show.** Ridge regression in practice: "we do not put prior on the intercept"; Step 1 mean-centre $\mathbf y$ and every column of $\mathbf X$; Step 2 fit the model without intercept, $\mathbf w = (\mathbf X_c^\top\mathbf X_c + \frac{\sigma^2}{\sigma_0^2}\mathbf I)^{-1}\mathbf X_c^\top\mathbf y_c$; Step 3 recover $w_0 = \text{mean}(y) - [\text{mean}(x_1)\dots\text{mean}(x_J)]\mathbf w$.
- **What this notebook adds.** A two-line proof that the recipe solves the problem with an unpenalised intercept, two independent numerical checks (penalty matrix $\operatorname{diag}(0,1,\dots,1)$; scikit-learn), a demonstration of what goes wrong when the intercept is penalised (biased predictions, genetic effects that change with the units of $y$), why features must be standardised for ridge but not for least squares, and the index typos in slide 58's $\mathbf X_c$.
- **Exam tip.** If asked "why don't we penalise $w_0$?": the intercept only sets the overall level of $y$; penalising it makes the estimates depend on where we put the zero of the $y$ scale. And always standardise with *training* statistics.

---
## 5 · Choosing $\lambda$ with a validation set — a GWAS cohort with linkage disequilibrium

> 📍 **Where we are.** $\lambda$ is not learned by minimising the training error: the training error always prefers
> $\lambda\to0$ (least squares fits the training data best). We need data the model has not seen. The simplest approach is
> a **validation set**: fit on training people, pick the $\lambda$ that predicts validation people best, and report the error
> on a separate **test** set that played no part in any choice. ($k$-fold cross-validation, which reuses the data more
> efficiently, gets a preview in Section 8 and a full treatment in Module 23.)

From here on we use a more realistic simulated cohort, with $J = 1{,}000$ SNPs, **more SNPs than training people**, and
**linkage disequilibrium**: SNPs come in 100 blocks of 10 neighbours whose genotypes are correlated, as along a real
chromosome. Inside a block, each chromosome copy is generated by a chain of correlated Gaussian "liabilities" (each
SNP's value is 0.95 × its neighbour's plus fresh noise) that are thresholded to give the allele; a person's genotype adds
the two copies. Ten blocks each contain one causal SNP; the heritability is 0.5. The cohort has 1,000 people: 300 for
training, 200 for validation, 500 for testing.

In [ ]:
# ── 5.1 A SIMULATED cohort with LD blocks: 1,000 people × 1,000 SNPs, 10 causal ──
def simulate_ld_genotypes(rng, n, n_blocks, block_size, rho, maf_range=(0.1, 0.5)):
    """0/1/2 genotypes with linkage disequilibrium inside blocks of neighbouring SNPs.

    For each chromosome copy, a Gaussian chain z_k = rho·z_(k−1) + sqrt(1 − rho²)·noise runs along each block; the
    allele is 1 when z_k falls below the quantile that gives allele frequency f_k. Genotype = sum of the two copies.
    """
    J = n_blocks * block_size
    f = rng.uniform(*maf_range, J)
    threshold = stats.norm.ppf(f)                       # P(z < threshold) = f
    G = np.zeros((n, J), dtype=int)
    for _ in range(2):                                  # two chromosome copies per person
        z = np.empty((n, J))
        for b in range(n_blocks):
            s = b * block_size
            z[:, s] = rng.normal(size=n)
            for k in range(1, block_size):
                z[:, s + k] = rho * z[:, s + k - 1] + np.sqrt(1 - rho ** 2) * rng.normal(size=n)
        G += (z < threshold).astype(int)
    return G, f


rng5 = np.random.default_rng(SEED + 9)
n_blocks, block_size = 100, 10
G5, f5 = simulate_ld_genotypes(rng5, 1000, n_blocks, block_size, rho=0.95)
J5 = G5.shape[1]
block = np.arange(J5) // block_size                     # which LD block each SNP belongs to
causal_blocks = rng5.choice(n_blocks, 10, replace=False)
causal5 = np.sort(causal_blocks * block_size + rng5.integers(0, block_size, 10))   # one causal SNP in each of 10 blocks
y5, w5_true, causal5 = simulate_quantitative_trait(rng5, G5, causal=causal5, h2=0.5)
tr5, va5, te5 = np.arange(300), np.arange(300, 500), np.arange(500, 1000)
is_causal5 = np.isin(np.arange(J5), causal5)
adjacent_r = [np.corrcoef(G5[:, j], G5[:, j + 1])[0, 1] for j in range(J5 - 1) if block[j] == block[j + 1]]
print(f"genotypes {G5.shape}; correlation between neighbouring SNPs in a block: median {np.median(adjacent_r):.2f}")
print("causal SNPs:", causal5, "\ntrue effects:", np.round(w5_true[causal5], 2))

fig, ax = plt.subplots(figsize=(4.4, 3.6))
im = ax.imshow(np.corrcoef(G5[:, :40].T), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xlabel("SNP (first 40 = 4 LD blocks)"); ax.set_ylabel("SNP"); fig.colorbar(im, ax=ax, label="correlation r")
ax.set_title("LD blocks: correlated neighbours (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The correlation map shows the block structure: SNPs inside a block are strongly correlated
(neighbours typically $r\approx0.7$), SNPs in different blocks are not. This is a cartoon of real LD, enough to create the
"passenger SNP" problem of Module 06. Now fit ridge on the 300 training people for a grid of $\lambda$ values.

In [ ]:
# ── 5.2 Ridge on 300 people × 1,000 SNPs: training and validation error over a grid of λ ──
X5_tr, X5_va, X5_te = (G5[idx].astype(float) for idx in (tr5, va5, te5))
lams5 = np.logspace(-1, 5, 25)
mse_tr5, mse_va5, fits5 = [], [], []
for lam in lams5:
    w0, w = fit_ridge(X5_tr, y5[tr5], lam)
    fits5.append((w0, w))
    mse_tr5.append(np.mean((y5[tr5] - w0 - X5_tr @ w) ** 2))
    mse_va5.append(np.mean((y5[va5] - w0 - X5_va @ w) ** 2))
k_best = int(np.argmin(mse_va5))
lam_ridge5 = lams5[k_best]
w0_rg5, w_rg5 = fits5[k_best]
mse_te_ridge5 = np.mean((y5[te5] - w0_rg5 - X5_te @ w_rg5) ** 2)
mse_te_mean5 = np.mean((y5[te5] - y5[tr5].mean()) ** 2)
print(f"λ chosen on the validation set: {lam_ridge5:.0f}")
print(f"test MSE (500 untouched people): ridge {mse_te_ridge5:.3f} | predict the mean {mse_te_mean5:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(lams5, mse_tr5, "o-", ms=3, label="training error (300 people)")
ax.plot(lams5, mse_va5, "s-", ms=3, label="validation error (200 people)")
ax.axvline(lam_ridge5, color="k", ls=":", lw=1, label=f"chosen λ = {lam_ridge5:.0f}")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("mean squared error"); ax.legend(fontsize=7)
ax.set_title("choose λ where the validation error is lowest")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The training error rises steadily with $\lambda$ — it would always choose the smallest $\lambda$,
i.e. near-perfect memorisation. The validation error is U-shaped: large at small $\lambda$ (overfitting 1,000 weights to 300
people), large at huge $\lambda$ (everything shrunk to zero ≈ predicting the mean), and lowest in between. On the 500 test
people, the chosen ridge model clearly beats predicting the mean — but, as we will see, the lasso does much better on this
cohort, because here the truth is sparse: 10 causal SNPs out of 1,000.

⚠️ **The test set must stay untouched.** If we had picked $\lambda$ by looking at test error, the reported test error would be
optimistically biased — the cardinal rule of Module 23.

✏️ **Your turn.** Replace the validation set by the *training* set when choosing $\lambda$ (`np.argmin(mse_tr5)`). Which
$\lambda$ is chosen, and what happens to the test MSE?

---
## 6 · Sparsity: from best subsets to the lasso (slides 60–68)

> 📍 **Where we are.** Ridge shrinks every weight but keeps all 1,000 SNPs in the model. Slides 60–68 ask a sharper
> question: *which SNPs are truly significant?* — and answer it with a different penalty.

**A family of penalties (slide 61).** Regularisation is a general recipe: MLE criterion plus a **regularisation function**
$F(\mathbf w)$, weighted by $\lambda$:

$$\underset{\mathbf w}{\arg\min}\ (\mathbf y-\mathbf X\mathbf w)^\top(\mathbf y-\mathbf X\mathbf w) + \lambda\,F(\mathbf w).$$

*In words:* fit the data, but pay for "complexity" as measured by $F$. Ridge uses $F(\mathbf w) = \mathbf w^\top\mathbf w = \sum_j w_j^2$;
the **lasso** ("least absolute shrinkage and selection operator") uses $F(\mathbf w) = \sum_j |w_j|$. $F$ encodes the prior
belief you want to impose, and a large $\lambda$ means more influence of that belief (slide 61).

**Why sparsity? (slides 62–65)** *Biologically*, each phenotype is likely to be associated with a small number of SNPs rather
than all of them — a single variant in Huntington's disease, a few genes for inherited breast-cancer risk. *Statistically*,
learning a few non-zero weights is an easier problem than learning many. And a short list of SNPs is *interpretable* (slide
65's notes: it is much easier to look at 5 genes than at 1,000).

**The ideal: count the non-zeros (slide 66).** Sparsity "in a mathematical sense" means

$$\hat{\mathbf w} = \underset{\mathbf w}{\arg\min}\ \lVert\mathbf y-\mathbf X\mathbf w\rVert^2\quad\text{subject to}\quad
\sum_{j=1}^{J}\mathbb I\big[|w_j| > 0\big] \le C ,$$

*In words:* find the best fit that uses at most $C$ SNPs. The **indicator function** $\mathbb I[A]$ is 1 if condition $A$ is true
and 0 otherwise, so the sum counts the non-zero weights (sometimes written $\lVert\mathbf w\rVert_0$, the "$L_0$ norm").

Note that slides 66–67 switch notation: they write the weights as $\boldsymbol\beta$ and the number of features as $p$ (and
$x_n$ in the diagram); we keep $\mathbf w$ and $J$. "But solving this problem is computationally intractable" (slide 66): there
is no shortcut around trying subsets, and there are $2^J$ of them. Let us feel that on the prostate data, where $J = 8$ is
still small enough to try all $2^8 = 256$ subsets.

In [ ]:
# ── 6.1 Best-subset selection by brute force on the prostate data (all 2^8 = 256 subsets) ──
t0 = time.perf_counter()
records = []
for k in range(0, 9):
    for subset in itertools.combinations(range(8), k):
        A = np.column_stack([np.ones(len(y_tr)), Z_tr[:, list(subset)]])
        w = np.linalg.lstsq(A, y_tr, rcond=None)[0]
        records.append((k, np.sum((y_tr - A @ w) ** 2), [predictors[j] for j in subset]))
seconds_per_fit = (time.perf_counter() - t0) / len(records)
best = {}
for k, rss_k, names in records:
    if k not in best or rss_k < best[k][0]:
        best[k] = (rss_k, names)
for k in range(9):
    print(f"best subset of size {k}: training RSS {best[k][0]:6.2f}  {best[k][1]}")
for J_big in [8, 15, 30, 60, 2000]:                  # work in log10 units: 2^2000 overflows a float
    log10_subsets = J_big * np.log10(2)
    log10_seconds = log10_subsets + np.log10(seconds_per_fit)
    print(f"J = {J_big:5d}: 2^J ≈ 10^{log10_subsets:.1f} subsets → about 10^{log10_seconds:.1f} s "
          f"= 10^{log10_seconds - np.log10(3.15e7):.1f} years at this speed")

fig, ax = plt.subplots(figsize=(5.5, 3.3))
ax.scatter([r[0] for r in records], [r[1] for r in records], s=8, color="0.6", label="all 256 subsets")
ax.plot(range(9), [best[k][0] for k in range(9)], "o-", color="tab:red", label="best subset of each size")
ax.set_xlabel("number of predictors in the model"); ax.set_ylabel("training RSS"); ax.legend(fontsize=7)
ax.set_title("best-subset selection, prostate (REAL)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** For each size we found the subset with the smallest training error: `lcavol` alone, then
`lcavol + lweight`, then `+ svi`, and so on. Here each best subset happens to extend the previous one, but that is not
guaranteed in general (the best pair need not contain the best single predictor), which is why no greedy shortcut is
guaranteed to find them. (Training RSS always decreases with size, so choosing the *size* again needs validation data.) With
8 predictors brute force takes a fraction of a second and with 15 a few seconds at most; with 60 it would take on the order of a million years
at the same speed (the exact figure depends on your computer), and with the 2,000 SNPs of Module 06 a number of years with
about 600 digits — a GWAS has millions of SNPs. The $L_0$ problem is hopeless at genomic scale.

### 6.1 The lasso: a convex relaxation (slide 67)

Replace the count of non-zeros by the **sum of absolute values**, $\lVert\mathbf w\rVert_1 = \sum_j |w_j|$:

$$\text{constrained form:}\quad \hat{\mathbf w} = \underset{\mathbf w}{\arg\min}\ \lVert\mathbf y-\mathbf X\mathbf w\rVert^2\ \ \text{s.t.}\ \sum_{j=1}^J|w_j|\le C
\qquad\qquad
\text{lasso form:}\quad \hat{\mathbf w} = \underset{\mathbf w}{\arg\min}\ \lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert_1 .$$

*In words:* either cap the total absolute size of the weights at a budget $C$, or charge $\lambda$ for every unit of absolute
weight. The two forms are equivalent — for every budget $C$ there is a $\lambda$ that gives the same solution (the
Lagrange-multiplier idea of Module 04) — and "this form of the problem is easier to solve!" because the objective is
**convex**: a single bowl with no local minima. Remarkably, unlike ridge, the lasso sets many weights **exactly** to zero.
Why? The geometry of the constraint tells the story.

In [ ]:
# ── 6.2 Why the L1 constraint gives exact zeros: diamond vs circle ──
w_hat2 = np.array([1.6, 0.3])                               # least-squares solution (outside both regions)
H = np.array([[1.0, 0.6], [0.6, 1.0]])                      # RSS(w) = (w − ŵ)ᵀ H (w − ŵ) + const (correlated features)


def rss_quadratic(W):
    """Residual sum of squares (up to a constant) for an array of candidate weight vectors W (rows)."""
    D = W - w_hat2
    return np.einsum("ij,jk,ik->i", D, H, D)


theta = np.linspace(0, 2 * np.pi, 20001)
circle = np.column_stack([np.cos(theta), np.sin(theta)])                     # boundary of w1² + w2² ≤ 1
diamond = circle / np.abs(circle).sum(axis=1, keepdims=True)                 # boundary of |w1| + |w2| ≤ 1
sol_l2, sol_l1 = circle[np.argmin(rss_quadratic(circle))], diamond[np.argmin(rss_quadratic(diamond))]
print("L2 (ridge-type) constrained solution:", np.round(sol_l2, 3))
print("L1 (lasso-type) constrained solution:", np.round(sol_l1, 3), "← lands on a corner: w2 = 0")

g = np.linspace(-1.5, 2.5, 300)
B1, B2 = np.meshgrid(g, g)
R = rss_quadratic(np.column_stack([B1.ravel(), B2.ravel()])).reshape(B1.shape)
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, boundary, sol, title in zip(axes, [diamond, circle], [sol_l1, sol_l2],
                                    ["lasso: |w1| + |w2| ≤ 1", "ridge: w1² + w2² ≤ 1"]):
    touch = rss_quadratic(sol[None])[0]                                     # the ellipse that touches the region
    ax.contour(B1, B2, R, levels=sorted({round(touch, 4), 0.1, 0.3, 1.5, 3.0}), colors="tab:blue", linewidths=1)
    ax.fill(boundary[:, 0], boundary[:, 1], color="tab:orange", alpha=0.35)
    ax.plot(*w_hat2, "b^", ms=8, label="least squares ŵ"); ax.plot(*sol, "r*", ms=14, label="constrained solution")
    ax.axhline(0, color="k", lw=0.5); ax.axvline(0, color="k", lw=0.5)
    ax.set_aspect("equal"); ax.set_xlabel("w_1"); ax.set_title(title); ax.legend(fontsize=7, loc="lower right")
axes[0].set_ylabel("w_2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The blue ellipses are contours of the squared error around the least-squares solution. The
constrained solution is where the smallest ellipse first touches the orange region. The $L_2$ disc is round, so the touching
point is generally *not* on an axis: both weights stay non-zero (ridge shrinks, does not select). The $L_1$ region is a
diamond with **corners on the axes**; an ellipse coming from outside very often touches it first at a corner, where one
weight is exactly zero — here $w_2 = 0$. In $J$ dimensions the $L_1$ ball has even more corners and edges lying in
coordinate planes, so many weights are zero at once (slide 67's notes: "the objective function will be minimized at a
corner of the feasible set, resulting in a sparse solution").

### 6.2 The lasso as a MAP estimate: the Laplace prior

Ridge was MAP with a Gaussian prior. Which prior gives the lasso? Try the **Laplace** (double-exponential) distribution,
independently for each weight:

$$p(w_j) = \frac{1}{2b}\exp\!\Big(-\frac{|w_j|}{b}\Big),\qquad b > 0 .$$

*In words:* a sharp peak at zero and tails that decay like $e^{-|w|}$ — more slowly than a Gaussian's $e^{-w^2}$; most
effects are believed to be tiny, a few may be large.

Repeat the MAP derivation of Section 2 with this prior: $\log p(\mathbf w) = -J\log(2b) - \frac1b\sum_j|w_j|$, so

$$\mathbf w_{MAP} = \underset{\mathbf w}{\arg\min}\ \Big[\frac{1}{2\sigma^2}\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \frac1b\sum_j|w_j|\Big]
= \underset{\mathbf w}{\arg\min}\ \Big[\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \frac{2\sigma^2}{b}\lVert\mathbf w\rVert_1\Big].$$

*In words:* drop the constant, flip the sign, multiply by $2\sigma^2$ — and the lasso appears with $\lambda = 2\sigma^2/b$. A
narrower Laplace prior (small $b$) means a larger $\lambda$ and more zeros.

In [ ]:
# ── 6.3 Gaussian vs Laplace prior with the same variance ──
s0 = 0.5
b = s0 / np.sqrt(2)                                     # Laplace variance is 2b², so this matches the Gaussian's σ0²
gw = np.linspace(-2.5, 2.5, 801)
fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.2))
for ax, scale in zip(axes, ["linear", "log"]):
    ax.plot(gw, stats.norm.pdf(gw, 0, s0), color="tab:blue", label=f"Gaussian, σ0 = {s0} → ridge")
    ax.plot(gw, stats.laplace.pdf(gw, 0, b), color="tab:red", label=f"Laplace, b = {b:.3f} → lasso")
    ax.set_yscale(scale); ax.set_xlabel("weight w_j"); ax.set_ylabel("prior density")
axes[0].legend(fontsize=7); axes[0].set_title("same variance, different shapes")
axes[1].set_ylim(1e-6, 5); axes[1].set_title("log scale: Laplace has heavier tails")
plt.tight_layout(); plt.show()
print(f"prior density at w = 0: Gaussian {stats.norm.pdf(0, 0, s0):.2f}, Laplace {stats.laplace.pdf(0, 0, b):.2f}")
print(f"prior probability that |w| > 1.5: Gaussian {2 * stats.norm.sf(1.5, 0, s0):.1e}, Laplace {2 * stats.laplace.sf(1.5, 0, b):.1e}")

🔍 **What just happened?** With the same variance, the Laplace prior puts more mass right at zero (a sharp peak) *and* more
mass far out (heavier tails, clearer on the log scale) — a good description of genetic architecture: most SNPs do nothing,
a few matter a lot. The Gaussian prior prefers many small, similar effects.

⚠️ **Where do the exact zeros come from?** From taking the **mode** (MAP) of the posterior: the sharp peak of the Laplace
prior makes the posterior's highest point sit exactly at zero for many weights. The posterior *mean* under a Laplace prior
(the "Bayesian lasso", Park & Casella 2008) is never exactly zero. Sparsity is a property of the estimator, not a statement
that the biology is exactly sparse.

✏️ **Your turn.** In cell 6.2 move the least-squares point to `w_hat2 = np.array([1.6, 1.2])` and re-run. Does the lasso
solution still land on a corner? What does that tell you about when the lasso keeps both features?

### 📑 Slide connection — `002_regression`, slides 60–68

- **What the slides show.** Regularisation as "MLE criterion + $\lambda F(\mathbf w)$" with ridge ($F = \mathbf w^\top\mathbf w$) and lasso ($F = \sum|w_i|$); the association-strength picture "which SNPs are truly significant?" and "many SNPs with zero effects (sparse results) — what prior should we use?"; why sparsity makes statistical and biological sense; the $L_0$ problem with an indicator function ("computationally intractable"); the lasso's constrained and penalised forms (Tibshirani 1996; Wu *et al.* 2009).
- **What this notebook adds.** Brute-force best subsets on the real prostate data with a cost extrapolation, the diamond-vs-circle geometry computed numerically, the Laplace prior derivation giving $\lambda = 2\sigma^2/b$ (the slides ask "what prior?" but do not answer), and the caveat that MAP sparsity comes from the posterior mode. We flag the slides' switch to $\boldsymbol\beta$, $p$ and $n$ on slides 66–67.
- **Exam tip.** Explain with a picture why $L_1$ gives zeros and $L_2$ does not (corners), and state the priors: Gaussian ↔ ridge, Laplace ↔ lasso. Know why $L_0$ is intractable ($2^J$ subsets, non-convex) and that the lasso is its convex relaxation.

---
## 7 · Solving the lasso: soft-thresholding and coordinate descent (slide 69)

> 📍 **Where we are.** Ridge had a closed form because its objective is smooth. The lasso objective has a **kink** at
> $w_j = 0$ — the absolute value $|w|$ has no derivative there — so "set the gradient to zero" fails. Slide 69 names the
> standard solver: **pathwise coordinate descent** (Friedman *et al.* 2007), which updates one $w_j$ at a time. Its heart is
> the one-dimensional lasso, which we can solve exactly by hand.

### 7.1 The one-dimensional lasso

Fix all weights except one. Its objective has the form (with $r_i$ the part of $y_i$ the other features do not explain)

$$f(w) = \sum_{i=1}^N (r_i - x_i w)^2 + \lambda|w| = a\,w^2 - 2c\,w + \sum_i r_i^2 + \lambda|w|,
\qquad a = \sum_i x_i^2,\ \ c = \sum_i x_i r_i .$$

*In words:* expanding the square gives a parabola in $w$ ($a > 0$ measures how much the feature varies, $c$ how strongly it
lines up with what is left to explain), plus the V-shaped penalty.

Away from zero, $|w|$ is smooth, so solve each side separately:

* **If $w > 0$:** $|w| = w$, so $f'(w) = 2aw - 2c + \lambda = 0$ gives $w = (c - \lambda/2)/a$. This is positive — a valid
  answer for this case — only if $c > \lambda/2$.
* **If $w < 0$:** $|w| = -w$, so $f'(w) = 2aw - 2c - \lambda = 0$ gives $w = (c + \lambda/2)/a$, valid only if $c < -\lambda/2$.
* **Otherwise** ($|c| \le \lambda/2$): neither side has a valley, so the minimum is at the kink, $w = 0$. Check: just left
  of 0 the slope is $-2c - \lambda \le 0$ (going down), just right of 0 it is $-2c + \lambda \ge 0$ (going up) — zero is the
  bottom. (The set of slopes $[-2c-\lambda,\, -2c+\lambda]$ at the kink is called the **subgradient**; "0 lies in it" is
  the optimality condition for non-smooth convex functions.)

All three cases fit in one formula, **soft-thresholding**:

$$\boxed{\ w^\star = \frac{S(c,\ \lambda/2)}{a},\qquad S(z, t) = \operatorname{sign}(z)\,\max\big(|z| - t,\ 0\big)\ }$$

*In words:* compute the least-squares answer's numerator $c$, move it toward zero by $\lambda/2$, and if it would cross zero,
stop at exactly zero. Compare ridge in one dimension: $f(w) = aw^2 - 2cw + \lambda w^2$ gives $w = c/(a+\lambda)$ — a
proportional shrink that never reaches zero.

In [ ]:
# ── 7.1 Soft-thresholding: check against brute force, and compare the three "shrinkage rules" ──
def soft_threshold(z, t):
    """S(z, t) = sign(z)·max(|z| − t, 0): move z toward 0 by t, stopping at 0."""
    return np.sign(z) * np.maximum(np.abs(z) - t, 0.0)


rng7 = np.random.default_rng(SEED + 7)
grid = np.linspace(-6, 6, 120001)                         # brute force on a fine grid (spacing 1e-4)
worst = 0.0
for _ in range(300):                                      # 300 random one-dimensional problems
    a, c, lam = rng7.uniform(1, 3), rng7.uniform(-5, 5), rng7.uniform(0, 6)   # |c/a| ≤ 5 stays inside the grid
    f_grid = a * grid ** 2 - 2 * c * grid + lam * np.abs(grid)
    worst = max(worst, abs(grid[np.argmin(f_grid)] - soft_threshold(c, lam / 2) / a))
print(f"largest |brute force − formula| over 300 random problems: {worst:.1e} (grid spacing 1e-4)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
wg = np.linspace(-2, 2, 401)
for c, col in zip([0.4, 1.0, 2.0], ["tab:green", "tab:orange", "tab:red"]):
    fw = wg ** 2 - 2 * c * wg + 2.0 * np.abs(wg)           # a = 1, λ = 2  ⇒  threshold λ/2 = 1
    axes[0].plot(wg, fw, color=col, label=f"c = {c}: minimum at w = {soft_threshold(c, 1.0):.1f}")
    axes[0].plot(soft_threshold(c, 1.0), np.min(fw), "o", color=col)
axes[0].set_xlabel("w"); axes[0].set_ylabel("f(w) = w² − 2cw + 2|w|"); axes[0].legend(fontsize=7)
axes[0].set_title("the kink at 0 traps the minimum when |c| ≤ λ/2")
z = np.linspace(-3, 3, 301)
axes[1].plot(z, z, "k--", lw=1, label="least squares: c/a")
axes[1].plot(z, z / (1 + 1.0), color="tab:blue", label="ridge: c/(a + λ)")
axes[1].plot(z, soft_threshold(z, 1.0), color="tab:red", lw=2, label="lasso: S(c, λ/2)/a")
axes[1].set_xlabel("c (with a = 1, λ = 1 for ridge, λ = 2 for lasso)"); axes[1].set_ylabel("estimate w*")
axes[1].legend(fontsize=7); axes[1].set_title("ridge shrinks proportionally; lasso shrinks and kills")
plt.tight_layout(); plt.show()

🔍 **What just happened?** On 300 random problems the formula agrees with brute-force minimisation to within the grid
spacing. Left: for small $c$ (green and orange, $|c| \le \lambda/2 = 1$) the V-shaped penalty traps the minimum at exactly
zero; for larger $c$ (red) the minimum moves off zero but is shifted by $\lambda/2a$. Right: the three rules side by side — the lasso's
soft-threshold has a **dead zone** $|c| \le \lambda/2$ where the estimate is exactly zero, and outside it every estimate is
pulled toward zero by the same amount (ESL Table 3.4 shows the same picture).

### 7.2 Coordinate descent

Now apply the one-dimensional solution to each weight in turn (slide 69: "in each iteration, update each $\beta_j$, and
iterate until convergence"). For feature $j$, the part of $\mathbf y$ that the *other* features leave unexplained is the
**partial residual** $\mathbf r^{(j)} = \mathbf y - \sum_{k\ne j}\mathbf x_k w_k$, where $\mathbf x_k$ is column $k$ of $\mathbf X$. Then

$$a_j = \mathbf x_j^\top\mathbf x_j,\qquad c_j = \mathbf x_j^\top\mathbf r^{(j)},\qquad w_j \leftarrow \frac{S(c_j,\ \lambda/2)}{a_j}.$$

*In words:* see how well feature $j$ alone explains what the others leave over; soft-threshold that; move on to the next
feature; sweep through all features again and again until nothing changes.

Each update *exactly* minimises the objective along one coordinate, so the objective can never increase. For the lasso
(convex, with a penalty that is a sum of one-variable terms) this simple scheme converges to the global minimum. A
bookkeeping trick makes it fast: keep the full residual $\mathbf r = \mathbf y - \mathbf X\mathbf w$ up to date; then
$\mathbf r^{(j)} = \mathbf r + \mathbf x_j w_j$, so $c_j = \mathbf x_j^\top\mathbf r + a_j w_j$.

**Where does the path start?** If all weights are zero, the residual is $\mathbf y$ itself and every update keeps $w_j = 0$ as long
as $|\mathbf x_j^\top\mathbf y| \le \lambda/2$. So for

$$\lambda \ge \lambda_{max} = 2\max_j\big|\mathbf x_j^\top\mathbf y\big|$$

*In words:* above this penalty, not a single feature is worth its price, and the lasso solution is $\mathbf w = \mathbf 0$ (for centred data).

In [ ]:
# ── 7.2 Lasso by cyclic coordinate descent, from scratch ──
def lasso_objective(w, X, y, lam):
    """The slides' lasso objective ||y − Xw||² + λ·Σ|w_j|."""
    r = y - X @ w
    return r @ r + lam * np.sum(np.abs(w))


def lasso_cd(X, y, lam, w_init=None, tol=1e-6, max_sweeps=500, track=False):
    """Minimise ||y − Xw||² + λ||w||₁ for CENTRED X and y by cyclic coordinate descent.

    Each update solves the one-dimensional lasso for w_j exactly (soft-thresholding) with the other weights fixed.
    Stops when no weight moves by more than `tol` in a full sweep. Returns (w, sweeps) or (w, sweeps, objectives).
    """
    X = np.asfortranarray(X, dtype=float)            # store columns contiguously: X[:, j] becomes fast
    w = np.zeros(X.shape[1]) if w_init is None else np.array(w_init, dtype=float)
    a = np.sum(X ** 2, axis=0)                       # a_j = x_jᵀx_j
    r = y - X @ w                                    # full residual, kept up to date
    history = [lasso_objective(w, X, y, lam)]
    for sweep in range(1, max_sweeps + 1):
        largest_step = 0.0
        for j in range(X.shape[1]):
            c_j = X[:, j] @ r + a[j] * w[j]          # x_jᵀ(partial residual without feature j)
            w_new = soft_threshold(c_j, lam / 2) / a[j]
            step = w_new - w[j]
            if step != 0.0:
                r -= step * X[:, j]
                w[j] = w_new
                largest_step = max(largest_step, abs(step))
        if track:
            history.append(lasso_objective(w, X, y, lam))
        if largest_step < tol:
            break
    return (w, sweep, history) if track else (w, sweep)

In [ ]:
# ── 7.2 (continued) Check on the prostate data: scikit-learn, monotone descent, λ_max ──
lam_max_psa = 2 * np.max(np.abs(Zc_tr.T @ yc_tr))
lam_p = 10.0
w_cd, sweeps, hist = lasso_cd(Zc_tr, yc_tr, lam_p, tol=1e-10, track=True)
sk_lasso = Lasso(alpha=lam_p / (2 * len(yc_tr)), tol=1e-12, max_iter=100000).fit(Z_tr, y_tr)   # α = λ / (2N)
print(f"prostate, λ = {lam_p}: converged in {sweeps} sweeps; objective never increased: {np.all(np.diff(hist) <= 1e-12)}")
print("our coordinate descent:", np.round(w_cd, 5))
print("scikit-learn Lasso    :", np.round(sk_lasso.coef_, 5))
print(f"λ_max = 2·max|x_jᵀy| = {lam_max_psa:.2f};  lasso at 1.001·λ_max gives all zeros: "
      f"{np.all(lasso_cd(Zc_tr, yc_tr, 1.001 * lam_max_psa)[0] == 0)}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(range(len(hist)), hist, "o-", ms=3)
ax.set_xlabel("sweep through all 8 coordinates"); ax.set_ylabel("lasso objective")
ax.set_title("coordinate descent only goes downhill")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our 25-line coordinate descent reproduces scikit-learn's lasso to five decimals, the objective
decreases at every sweep, and at $\lambda$ just above $\lambda_{max}$ every coefficient is exactly zero, as derived. At
$\lambda = 10$ three of the eight prostate predictors are already exactly zero (`age`, `lcp` and `gleason`).

⚠️ **Mind the scaling when you use a library.** scikit-learn's `Lasso(alpha=α)` minimises
$\frac{1}{2N}\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \alpha\lVert\mathbf w\rVert_1$. Multiplying by $2N$ shows it equals the slides'
objective with $\lambda = 2N\alpha$, i.e. $\alpha = \lambda/(2N)$. Forgetting this factor is the most common bug when comparing
lasso results across software (`glmnet` in R, slide 69, uses the same $\frac{1}{2N}$ convention).

### 7.3 Pathwise coordinate descent with warm starts

In practice we want the whole **regularisation path** — solutions for a decreasing sequence of $\lambda$ values, from
$\lambda_{max}$ down — because the best $\lambda$ is chosen afterwards. Solutions for neighbouring $\lambda$'s are close, so
start each fit from the previous solution (a **warm start**); this is the "pathwise" in slide 69's algorithm. Let us run
it on the 1,000-SNP cohort.

In [ ]:
# ── 7.3 The lasso path on 300 people × 1,000 SNPs, with warm starts ──
mu5 = X5_tr.mean(axis=0)
X5c, y5c = X5_tr - mu5, y5[tr5] - y5[tr5].mean()
lam_max5 = 2 * np.max(np.abs(X5c.T @ y5c))
lams_l5 = lam_max5 * np.logspace(0, -1.3, 25)          # from λ_max down to λ_max / 20
t0 = time.perf_counter()
w = np.zeros(J5); path5, sweeps5 = [], []
for lam in lams_l5:
    w, s = lasso_cd(X5c, y5c, lam, w_init=w, tol=1e-5)  # warm start from the previous solution
    path5.append(w.copy()); sweeps5.append(s)
path5 = np.array(path5)
t_path = time.perf_counter() - t0
nnz5 = (path5 != 0).sum(axis=1)
print(f"25 λ values in {t_path:.1f} s; sweeps per λ: {sweeps5}")
print("number of selected SNPs along the path:", nnz5)
t0 = time.perf_counter()
sweeps_cold = [lasso_cd(X5c, y5c, lam, tol=1e-5)[1] for lam in lams_l5]   # the same path, each fit started from 0
print(f"cold starts: {time.perf_counter() - t0:.1f} s; sweeps per λ: {sweeps_cold}")
print(f"total sweeps over the path: warm starts {sum(sweeps5)}, cold starts {sum(sweeps_cold)}")
k_mid = 12
sk5 = Lasso(alpha=lams_l5[k_mid] / (2 * len(tr5)), tol=1e-10, max_iter=100000).fit(X5_tr, y5[tr5])
print(f"max |ours − scikit-learn| at that λ: {np.abs(sk5.coef_ - path5[k_mid]).max():.1e}")

fig, ax = plt.subplots(figsize=(7, 3.6))
for j in np.flatnonzero((path5 != 0).any(axis=0)):
    ax.plot(lams_l5, path5[:, j], color="tab:red" if is_causal5[j] else "0.7", lw=1.6 if is_causal5[j] else 0.8)
ax.set_xscale("log"); ax.invert_xaxis(); ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("λ (decreasing →)"); ax.set_ylabel("lasso weight"); ax.plot([], [], color="tab:red", label="causal SNPs")
ax.plot([], [], color="0.7", label="null SNPs"); ax.legend(fontsize=7)
ax.set_title("lasso path, SIMULATED GWAS (300 people, 1,000 SNPs)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** At $\lambda_{max}$ no SNP is selected; as $\lambda$ decreases, SNPs enter one by one, and the
first to enter — the red curves — are mostly the truly causal ones. Further down the path more and more null SNPs (grey)
join with small weights. Smaller $\lambda$ means more non-zero weights and more sweeps to converge. Warm starts help only
modestly here (roughly a fifth fewer sweeps over the whole path, mostly at the small-$\lambda$ end); production solvers such
as `glmnet` combine them with further tricks, e.g. sweeping mainly over the currently non-zero weights. Our answer agrees
with scikit-learn's to about the convergence tolerance.

✏️ **Your turn.** Extend the path further down (`np.logspace(0, -2, 25)`) and watch the sweeps per $\lambda$ and the number
of selected SNPs. Why does the lasso never select more than about $N = 300$ SNPs, even when $\lambda$ is tiny? (Hint: think
about how many equations the training data provide.)

### 📑 Slide connection — `002_regression`, slides 69–70

- **What the slides show.** Estimating the lasso by the pathwise coordinate descent algorithm of Friedman *et al.* (Annals of Applied Statistics 2007): "in each iteration, update each $\beta_j$, and iterate until convergence"; a fast implementation is the R package `glmnet`. Slide 70 summarises: linear regression — MLE for the least-squared-error estimate; with a prior — MAP estimation, ridge regression if $N<J$, lasso regression for a sparse estimate.
- **What this notebook adds.** The update rule the slide leaves out: the full case analysis for the one-dimensional lasso, soft-thresholding $S(c,\lambda/2)/a$, the subgradient condition, $\lambda_{max} = 2\max_j|\mathbf x_j^\top\mathbf y|$, a from-scratch implementation with residual updates and warm starts, a check that the objective only decreases, and agreement with scikit-learn once its $\frac{1}{2N}$ scaling is accounted for ($\alpha = \lambda/2N$).
- **Exam tip.** Derive the soft-threshold update for the slides' objective $\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert_1$ — the threshold is $\lambda/2$ because the squared error has no ½ in front (with a ½ it would be $\lambda$). Know why the answer is exactly zero when $|c| \le \lambda/2$.

---
## 8 · Prostate cancer again: lasso path and cross-validated error

> 📍 **Where we are.** We have working ridge and lasso solvers. Back to the real data of Module 06 (slides 41–43): which
> clinical measures does the lasso keep, and do ridge or lasso predict PSA in new men better than least squares (test MSE
> 0.521)?

ESL (Fig. 3.10) plots the lasso path against the **shrinkage factor** $s = \lVert\hat{\mathbf w}(\lambda)\rVert_1 / \lVert\hat{\mathbf w}_{OLS}\rVert_1$,
which runs from 0 (all weights zero) to 1 (least squares). We do the same.

In [ ]:
# ── 8.1 The lasso path of the prostate data ──
lams_p = lam_max_psa * np.logspace(0, -3, 60)
w = np.zeros(8); path_p = []
for lam in lams_p:
    w, _ = lasso_cd(Zc_tr, yc_tr, lam, w_init=w, tol=1e-10)
    path_p.append(w.copy())
path_p = np.array(path_p)
w_ols_p = np.linalg.lstsq(Zc_tr, yc_tr, rcond=None)[0]
s_factor = np.abs(path_p).sum(axis=1) / np.abs(w_ols_p).sum()
first_in = [np.argmax(path_p[:, j] != 0) for j in range(8)]
print("order in which predictors enter:", [predictors[j] for j in np.argsort(first_in, kind="stable")])
print("lasso at the smallest λ:", np.round(path_p[-1], 3), "\nleast squares          :", np.round(w_ols_p, 3))

fig, ax = plt.subplots(figsize=(7, 3.8))
for j, name in enumerate(predictors):
    ax.plot(s_factor, path_p[:, j], lw=1.5, label=name)
ax.axhline(0, color="k", lw=0.5); ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.set_xlabel("shrinkage factor s = ||w(λ)||₁ / ||w_OLS||₁"); ax.set_ylabel("coefficient (per SD)")
ax.set_xlim(0, 1.0); ax.set_title("lasso path, prostate data (REAL, 67 training men)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Reading from left (heavy penalty) to right: `lcavol` enters first, followed by `lweight` and
`svi`, then `lbph`, `pgg45`, `age`, `lcp` and `gleason`; at $s = 1$ the lasso coincides with least squares. In contrast to the ridge path of Section 3,
coefficients are **exactly zero** until they enter — the lasso is a continuous version of best-subset selection, and its
first five entries even match the best subsets of sizes 1–5 found by brute force in Section 6.1 (this agreement is
not guaranteed in general). `age` and `lcp` enter late with negative signs, and `gleason` only at the very end.

Which $\lambda$ should we use? With only 67 training men, a single validation split would be very noisy, so we use
**10-fold cross-validation** (a preview of Module 23): split the 67 men into 10 groups, fit on 9 groups, measure the error
on the 10th, rotate, and average. Standardisation is recomputed inside every fold so that the held-out men never influence
the fit.

In [ ]:
# ── 8.2 10-fold cross-validation for ridge and lasso on the training men; then test once ──
def fit_lasso(X, y, lam, w_init=None):
    """Lasso with an unpenalised intercept: centre, run coordinate descent, recover w0. Returns (w0, w)."""
    x_bar, y_bar = X.mean(axis=0), y.mean()
    w, _ = lasso_cd(X - x_bar, y - y_bar, lam, w_init=w_init, tol=1e-8)
    return y_bar - x_bar @ w, w


rng8 = np.random.default_rng(SEED + 8)
fold = rng8.permutation(len(y_tr)) % 10                   # fold label 0–9 for each training man
X_tr_raw = X_raw[is_train]
grids = {"ridge": np.logspace(-2, 3, 30), "lasso": lam_max_psa * np.logspace(0, -3, 30)}
cv_mse = {name: np.zeros(len(grid)) for name, grid in grids.items()}
for k in range(10):
    inner, held = fold != k, fold == k
    Zi, m_i, s_i = standardize(X_tr_raw[inner])           # statistics of the 9 training folds only
    Zh, _, _ = standardize(X_tr_raw[held], m_i, s_i)
    w_warm = None
    for g, lam in enumerate(grids["lasso"]):
        w0, w_warm = fit_lasso(Zi, y_tr[inner], lam, w_init=w_warm)
        cv_mse["lasso"][g] += np.sum((y_tr[held] - w0 - Zh @ w_warm) ** 2) / len(y_tr)
    for g, lam in enumerate(grids["ridge"]):
        w0, w = fit_ridge(Zi, y_tr[inner], lam)
        cv_mse["ridge"][g] += np.sum((y_tr[held] - w0 - Zh @ w) ** 2) / len(y_tr)

rows = {}
w0_ols, w_ols_full = fit_ridge(Z_tr, y_tr, 1e-10)
rows["least squares"] = (np.mean((y_te - w0_ols - Z_te @ w_ols_full) ** 2), 8, "—")
for name, fit in [("ridge", fit_ridge), ("lasso", fit_lasso)]:
    lam_best = grids[name][np.argmin(cv_mse[name])]
    w0, w = fit(Z_tr, y_tr, lam_best)                     # refit on all 67 training men
    rows[name] = (np.mean((y_te - w0 - Z_te @ w) ** 2), int(np.sum(w != 0)), f"{lam_best:.3g}")
    if name == "lasso":
        print("lasso keeps:", [p for p, wj in zip(predictors, w) if wj != 0], "| drops:", [p for p, wj in zip(predictors, w) if wj == 0])
rows["predict the training mean"] = (np.mean((y_te - y_tr.mean()) ** 2), 0, "—")
print(pd.DataFrame(rows, index=["test MSE (30 men)", "non-zero coefficients", "λ chosen by 10-fold CV"]).T.to_string())

fig, ax = plt.subplots(figsize=(6, 3.3))
ax.plot(grids["ridge"], cv_mse["ridge"], "o-", ms=3, label="ridge")
ax.plot(grids["lasso"], cv_mse["lasso"], "s-", ms=3, label="lasso")
ax.set_xscale("log"); ax.set_xlabel("λ"); ax.set_ylabel("10-fold CV error"); ax.legend(fontsize=7)
ax.set_title("cross-validation curves (training men only)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Both CV curves are flat over a wide range of small $\lambda$ and rise at large $\lambda$; their minima
are shallow. Refitted on all 67 training men and evaluated once on the 30 test men, ridge and lasso both do slightly better
than least squares (test MSE ≈ 0.50 vs 0.521), and the lasso does it while dropping a predictor. With 8 predictors and 67
men, least squares is not in serious trouble, so regularisation brings only a small gain here; the large gains come when
$J$ is comparable to or larger than $N$, as in the GWAS cohort. And with 30 test men, differences of a few hundredths are
within noise — do not over-read them.

✏️ **Your turn.** Change the fold assignment seed (`SEED + 8` → `SEED + 18`) and re-run. Do the chosen $\lambda$ values and the
set of predictors kept by the lasso change? What does that tell you about selection with 67 patients?

---
## 9 · GWAS with the lasso: which SNPs are truly significant?

> 📍 **Where we are.** The model-selection extra slides 1–2 (the same pictures as slides 62–63) contrast multivariate
> regression, where "all SNPs are having impact on the phenotype", with regularised regression that gives "many SNPs with
> zero effects (sparse results)". We now compare three ways of finding causal SNPs on our 1,000-SNP cohort, where we know
> the answer: the univariate scan with Bonferroni correction (Module 06), ridge, and the lasso.

To score a selected set of SNPs we use **precision** (what fraction of the selected SNPs are causal), **recall** (what
fraction of the causal SNPs were selected) and their harmonic mean **F1** (toolkit `precision_recall_f1`; Module 09 explains
them in detail). Because of LD, we also report a **locus-level** recall: the fraction of causal *blocks* in which at least one
SNP was selected.

In [ ]:
# ── 9.1 Univariate scan vs ridge vs lasso for finding the 10 causal SNPs ──
def gwas_scan_p(G, y):
    """Two-sided p-values of the univariate least-squares slope for every SNP (Module 06, Section 5)."""
    X = G - G.mean(axis=0); yc = y - y.mean(); n = len(y)
    sxx = np.sum(X ** 2, axis=0)
    w = X.T @ yc / sxx
    se = np.sqrt((np.sum(yc ** 2) - w ** 2 * sxx) / (n - 2) / sxx)
    return 2 * stats.t.sf(np.abs(w / se), df=n - 2)


val_mse_l5 = [np.mean((y5[va5] - (y5[tr5].mean() - mu5 @ w) - X5_va @ w) ** 2) for w in path5]
k_l = int(np.argmin(val_mse_l5)); w_l5 = path5[k_l]; w0_l5 = y5[tr5].mean() - mu5 @ w_l5
k_sparse = int(np.argmax(nnz5 >= 10))                    # the first point on the path with at least 10 SNPs
p_uni = gwas_scan_p(X5_tr, y5[tr5])
uni_hits = p_uni < 0.05 / J5
sets = {"univariate + Bonferroni": uni_hits,
        "ridge: top 10 |w|": np.isin(np.arange(J5), np.argsort(-np.abs(w_rg5))[:10]),
        f"lasso, λ by validation ({nnz5[k_l]} SNPs)": w_l5 != 0,
        f"lasso, first λ with ≥ 10 SNPs": path5[k_sparse] != 0}
table = {}
for name, sel in sets.items():
    prec, rec, f1 = precision_recall_f1(is_causal5, sel)
    table[name] = {"selected": int(sel.sum()), "causal among them": int((sel & is_causal5).sum()),
                   "precision": prec, "recall": rec, "F1": f1,
                   "locus recall": len(set(block[sel]) & set(block[causal5])) / 10}
print(pd.DataFrame(table).T.round(2).to_string())
w_refit = np.linalg.lstsq(np.column_stack([np.ones(len(tr5)), X5_tr[:, uni_hits]]), y5[tr5], rcond=None)[0]
pred_uni = np.column_stack([np.ones(len(te5)), X5_te[:, uni_hits]]) @ w_refit
print(f"\ntest MSE: lasso (validation λ) {np.mean((y5[te5] - w0_l5 - X5_te @ w_l5) ** 2):.3f} | ridge {mse_te_ridge5:.3f} | "
      f"least squares on the univariate hits {np.mean((y5[te5] - pred_uni) ** 2):.3f} | mean {mse_te_mean5:.3f}")
print("univariate hits that are NOT causal:", [f"SNP {j} (block {block[j]}, causal SNP of that block: "
      f"{causal5[block[causal5] == block[j]].tolist()})" for j in np.flatnonzero(uni_hits & ~is_causal5)])

🔍 **What just happened?** Read the table row by row.

* **Univariate + Bonferroni** finds most causal SNPs with few false positives — and its false positives are *neighbours* of
  causal SNPs in the same LD block (printed below the table): passengers that ride along with the true variant, exactly the
  Module 06 puzzle.
* **Ridge** does not select; taking its 10 largest weights finds only some of the causal SNPs, because ridge spreads each
  causal effect over the correlated neighbours.
* **Lasso with $\lambda$ chosen for prediction** recovers all 10 causal SNPs (recall 1) but drags in dozens of null SNPs (low
  precision). This is a well-known property: the $\lambda$ that predicts best is smaller than the $\lambda$ that selects best,
  because a little extra signal from many small weights helps prediction.
* **Lasso at a sparser $\lambda$** (the first point on the path with at least 10 SNPs) gives a short list with high precision.

For **prediction**, the lasso clearly beats ridge on this sparse architecture, and also beats refitting least squares on the
univariate hits. (Had we simulated thousands of tiny effects — a highly polygenic trait — ridge would be the better
predictor; the best penalty matches the biology.)

In [ ]:
# ── 9.2 Zoom into LD blocks: how each method spreads one causal effect ──
show_blocks = list(block[uni_hits & ~is_causal5])[:2]            # blocks where the scan flagged a passenger SNP
show_blocks += [b for b in block[causal5] if b not in show_blocks][:2 - len(show_blocks)]
fig, axes = plt.subplots(1, len(show_blocks), figsize=(11, 3.2))
for ax, b in zip(np.atleast_1d(axes), show_blocks):
    snps = np.flatnonzero(block == b); xpos = np.arange(len(snps))
    ax.bar(xpos - 0.3, w5_true[snps], 0.2, color="k", label="true w")
    ax.bar(xpos - 0.1, w_rg5[snps], 0.2, color="tab:blue", label="ridge (validation λ)")
    ax.bar(xpos + 0.1, w_l5[snps], 0.2, color="tab:red", label="lasso (validation λ)")
    ax.bar(xpos + 0.3, path5[k_sparse][snps], 0.2, color="tab:orange", label="lasso (sparser λ)")
    lo, hi = ax.get_ylim()
    y_star = 0.9 * (hi if w5_true[snps].sum() > 0 else lo)          # put the stars on the side of the effect
    for x, j in zip(xpos, snps):
        if uni_hits[j]:
            ax.text(x, y_star, "★", ha="center", va="center", color="tab:green", fontsize=12)
    ax.axhline(0, color="k", lw=0.5); ax.set_xticks(xpos); ax.set_xticklabels(snps, fontsize=7, rotation=45)
    ax.set_xlabel(f"SNPs of LD block {b} (★ = univariate hit)"); ax.set_ylabel("weight")
np.atleast_1d(axes)[0].legend(fontsize=6); fig.suptitle("one causal SNP per block (SIMULATED)", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Inside an LD block the true effect sits on a single SNP (black). Ridge (blue) smears it over
several correlated neighbours with small weights. The lasso (red, orange) concentrates it on one or two SNPs — usually the
causal one, sometimes with a neighbour; in block 46 the sparser lasso (orange) even puts more weight on the passenger SNP
465 than on the causal SNP 466. The univariate scan (★) can flag several SNPs of the block because each of them,
on its own, correlates with the phenotype. No method can be certain *which* SNP of a tightly linked block is causal: that
requires fine-mapping, functional experiments, or data from populations with different LD patterns.

### 📑 Slide connection — `015_model-selection--extra`, slides 1–2

- **What the slides show.** Two association-strength pictures (genotype × $\boldsymbol\beta$ = trait): ordinary multivariate regression $\arg\min(\mathbf y-\mathbf X\boldsymbol\beta)^\top(\mathbf y-\mathbf X\boldsymbol\beta)$, where "all SNPs are having impact on the phenotypes: which SNPs are truly significant?", versus multivariate regression with regularisation, giving "many SNPs with zero effects (sparse results)" (Tibshirani 1996; Wu *et al.* 2009). They repeat slides 62–63 of `002_regression` as the opening of the model-selection lecture.
- **What this notebook adds.** A quantitative version of the picture on a simulated cohort with LD where the truth is known: precision, recall and locus-level recall for univariate tests, ridge and lasso; the observation that the prediction-optimal $\lambda$ over-selects; and a zoom into LD blocks showing how ridge smears and the lasso concentrates an effect. The extra deck's later slides (regularisation ↔ bias/variance, choosing $\lambda$ by cross-validation) are developed in Module 23.
- **Exam tip.** Ridge vs lasso in one sentence each: ridge keeps all SNPs and shrinks them (good when many small effects); lasso sets most to zero (good when few effects) but may pick an arbitrary member of a group of correlated SNPs.

---
## 10 · Elastic net, instability, and why a selected SNP is not a cause

> 📍 **Where we are.** The lasso gives short, readable lists. Before trusting such a list, ask two questions: *would I get
> the same list from a slightly different sample?* and *does selection mean causation?*

**Correlated SNPs.** When two SNPs are almost copies of each other, the lasso's objective barely changes if weight is moved
from one to the other, so which one it keeps can hinge on a few individuals. The **elastic net** (Zou & Hastie 2005) adds a
small ridge penalty to the lasso,

$$\underset{\mathbf w}{\arg\min}\ \lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda_1\lVert\mathbf w\rVert_1 + \lambda_2\lVert\mathbf w\rVert_2^2 ,$$

*In words:* the $L_1$ part still produces zeros, while the $L_2$ part makes the solution prefer to share weight among
strongly correlated features instead of picking one at random (the "grouping effect").

In scikit-learn, `ElasticNet(alpha=α, l1_ratio=ρ)` minimises $\frac{1}{2N}\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \alpha\rho\lVert\mathbf w\rVert_1 +
\frac{\alpha(1-\rho)}{2}\lVert\mathbf w\rVert_2^2$, i.e. $\lambda_1 = 2N\alpha\rho$ and $\lambda_2 = N\alpha(1-\rho)$ in our scaling.

Let us resample the people of a small study 200 times (the **bootstrap**: draw $N$ people *with replacement*) and see which of
two tightly linked SNPs each method keeps.

In [ ]:
# ── 10.1 Two tightly linked SNPs (only A is causal): lasso vs elastic net over 200 bootstrap samples ──
rng10 = np.random.default_rng(SEED + 10)
n10 = 300
snp_a = rng10.binomial(2, 0.3, n10)
snp_b = np.where(rng10.random(n10) < 0.95, snp_a, rng10.binomial(2, 0.3, n10))   # B copies A in 95 % of people
X10 = np.column_stack([snp_a, snp_b]).astype(float)
y10 = 0.4 * snp_a + rng10.normal(0, 1, n10)
print(f"correlation of A and B: {np.corrcoef(snp_a, snp_b)[0, 1]:.3f}")
alpha10 = 0.03
counts = {"A only": 0, "B only": 0, "both": 0, "neither": 0}
enet_w = []
for _ in range(200):
    boot = rng10.integers(0, n10, n10)                                  # resample people with replacement
    c = Lasso(alpha=alpha10).fit(X10[boot], y10[boot]).coef_
    key = "both" if (c != 0).all() else "A only" if c[0] != 0 else "B only" if c[1] != 0 else "neither"
    counts[key] += 1
    enet_w.append(ElasticNet(alpha=alpha10, l1_ratio=0.3).fit(X10[boot], y10[boot]).coef_)
enet_w = np.array(enet_w)
print("lasso — which SNPs are selected in 200 bootstrap samples:", counts)
print(f"elastic net — mean weight A {enet_w[:, 0].mean():.3f}, B {enet_w[:, 1].mean():.3f}; "
      f"both non-zero in {np.mean((enet_w != 0).all(axis=1)) * 100:.0f} % of samples")

🔍 **What just happened?** The lasso usually keeps the causal SNP A, but in some bootstrap samples it keeps **only the
passenger B**, and in others both: with $r\approx0.97$, a handful of people decide. The elastic net instead keeps both SNPs
in most samples and gives A the larger weight on average — a more honest summary of "the signal is somewhere in this pair".
Neither method can tell from these data alone which SNP is causal.

The same instability affects a whole GWAS. Let us bootstrap the 300 training people of our cohort 50 times, run the lasso
at the validation-chosen $\lambda$, and count how often each SNP is selected (the idea behind *stability selection*,
Meinshausen & Bühlmann 2010).

In [ ]:
# ── 10.2 Selection stability across 50 bootstrap samples of the GWAS cohort ──
alpha_l5 = lams_l5[k_l] / (2 * len(tr5))                               # our λ → scikit-learn's α
freq = np.zeros(J5)
for _ in range(50):
    boot = rng10.integers(0, len(tr5), len(tr5))
    freq += Lasso(alpha=alpha_l5, max_iter=5000).fit(X5_tr[boot], y5[tr5][boot]).coef_ != 0
freq /= 50
print("selection frequency of the 10 causal SNPs:", np.round(freq[causal5], 2))
print(f"null SNPs selected in ≥ 50 % of samples: {np.sum(freq[~is_causal5] >= 0.5)}; "
      f"in ≥ 10 %: {np.sum(freq[~is_causal5] >= 0.1)}; ever selected: {np.sum(freq[~is_causal5] > 0)} of {J5 - 10}")
in_causal_block = np.isin(block, block[causal5]) & ~is_causal5
print(f"mean selection frequency of null SNPs inside causal blocks: {freq[in_causal_block].mean():.2f}, "
      f"outside causal blocks: {freq[~np.isin(block, block[causal5])].mean():.2f}")
for j in causal5[freq[causal5] < 0.5]:                                 # causal SNPs that were often missed
    mates = np.flatnonzero((block == block[j]) & (np.arange(J5) != j))
    m = mates[np.argmax(freq[mates])]
    print(f"causal SNP {j} (selected {freq[j]:.0%}): its block-mate SNP {m} was selected {freq[m]:.0%} "
          f"(genotype correlation {np.corrcoef(G5[tr5, j], G5[tr5, m])[0, 1]:.2f})")

fig, ax = plt.subplots(figsize=(11, 3.0))
ax.bar(np.arange(J5), freq, width=1.0, color=np.where(is_causal5, "tab:red", np.where(in_causal_block, "tab:orange", "0.6")))
ax.set_xlabel("SNP (red = causal, orange = same LD block as a causal SNP, grey = other)")
ax.set_ylabel("selection frequency"); ax.set_xlim(0, J5)
ax.set_title("how often the lasso selects each SNP across 50 bootstrap samples (SIMULATED)")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Eight of the ten causal SNPs are selected in at least 94 % of the bootstrap samples — a stable
signal. The other two are selected in only 30–40 % of samples. For SNP 524 the lasso's choice wanders between it and its
block-mate 525 (selected in 50 % of samples, genotype correlation 0.80) — exactly the behaviour of the two-SNP example;
SNP 794 is simply selected less reliably at this sample size (its most frequent block-mate is chosen in only 14 %). Because $\lambda$ was tuned for
prediction, each fit also keeps dozens of null SNPs, and over 50 resamples more than half of all null SNPs are picked at
least once; a dozen or so reach 50 %, so frequency alone does not separate signal from noise perfectly here. Null SNPs
inside causal blocks (orange) are selected about twice as often as other null SNPs. A SNP that appears in one analysis and
vanishes in the next should not be reported as a discovery; selection frequencies (ideally at a sparser $\lambda$) are far
more honest than a single list.

⚠️ **Selected ≠ causal.** A lasso selection is a statistical statement about prediction *within this model and this sample*:

1. **LD:** the selected SNP may be a passenger correlated with the causal variant (Sections 9–10).
2. **Confounding:** population structure or environment can create associations without any genetic effect (Module 06,
   Section 5.5); the lasso does not fix this — adjustment (e.g. principal components, Module 14) is still needed.
3. **Shrinkage bias:** lasso weights of selected SNPs are pulled toward zero, so effect sizes are underestimated; refitting
   least squares on the selected set (the "relaxed lasso") is a common partial fix.
4. **No free $p$-values:** ordinary $p$-values computed *after* selecting SNPs on the same data are too optimistic; valid
   post-selection inference needs special methods or independent replication data.

✏️ **Your turn.** In cell 10.1 raise the copy probability from `0.95` to `0.99` and re-run. How do the lasso counts change?
Then set `l1_ratio=0.9` in the elastic net: does it behave more like the lasso or like ridge?

## 📌 Key takeaways

- **Linear regression with a prior is MAP estimation** (slide 70). With a Gaussian prior $\mathcal N(\mathbf 0,\sigma_0^2\mathbf I)$ the
  MAP estimate is **ridge regression**, $\mathbf w = (\mathbf X^\top\mathbf X + \lambda\mathbf I)^{-1}\mathbf X^\top\mathbf y$ with
  $\lambda = \sigma^2/\sigma_0^2$ — the same formula a frequentist calls $L_2$ regularisation.
- **Ridge works when $N<J$:** adding $\lambda\mathbf I$ raises every eigenvalue by $\lambda$, so the matrix is invertible; small
  $\sigma_0^2$ ⇔ large $\lambda$ ⇔ stronger shrinkage, more bias, less variance.
- **In practice:** centre the data, leave the intercept unpenalised and recover $w_0 = \bar y - \bar{\mathbf x}^\top\mathbf w$,
  standardise features with training statistics, and choose $\lambda$ on data not used for fitting (validation / CV).
- **Sparsity** is biologically and statistically sensible; the $L_0$ count of non-zeros is intractable ($2^J$ subsets), and the
  **lasso** $\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert_1$ is its convex relaxation — MAP with a Laplace prior,
  whose corners produce exact zeros.
- **The lasso is solved by coordinate descent:** each weight is updated by soft-thresholding, $w_j \leftarrow S(c_j,\lambda/2)/a_j$,
  along a decreasing $\lambda$ path with warm starts (slide 69); mind library scalings (scikit-learn $\alpha = \lambda/2N$).
- **Ridge vs lasso:** ridge for many small effects, lasso for few large ones; on our sparse GWAS the lasso predicted far better
  and found the causal loci — but the prediction-optimal $\lambda$ over-selects.
- **Selected ≠ causal:** correlated SNPs make lasso selections unstable (elastic net, bootstrap frequencies help), and LD,
  confounding and shrinkage bias all stand between a selected SNP and a biological mechanism.

---
## ✅ Checkpoint

1. A colleague writes the MAP estimate under the prior $w_j\sim\mathcal N(0,\sigma_0^2)$ as $(\mathbf X^\top\mathbf X + \frac{\sigma_0^2}{\sigma^2}\mathbf I)^{-1}\mathbf X^\top\mathbf y$. How can you tell, without redoing the algebra, that this is wrong?
   <details><summary>Answer</summary>Check a limit. A very vague prior ($\sigma_0\to\infty$) must give back least squares, i.e. the added term must vanish. With $\sigma_0^2/\sigma^2$ it would blow up instead. The correct ratio is $\lambda = \sigma^2/\sigma_0^2$.</details>
2. $\mathbf X^\top\mathbf X$ has eigenvalues 9, 4, 0, 0. What are the eigenvalues of $\mathbf X^\top\mathbf X + 2\mathbf I$, and is it invertible?
   <details><summary>Answer</summary>11, 6, 2, 2 — every eigenvalue is raised by 2 (same eigenvectors). None is zero, so the matrix is invertible (determinant $11\cdot6\cdot2\cdot2 = 264$).</details>
3. With orthonormal features ($\mathbf X^\top\mathbf X = \mathbf I$), the least-squares coefficients are (3.0, −0.8, 0.4). Give the ridge estimate for $\lambda = 1$ and the lasso estimate for $\lambda = 2$ (slides' scaling).
   <details><summary>Answer</summary>Ridge: divide by $1+\lambda = 2$ → (1.5, −0.4, 0.2). Lasso: here $a_j = 1$ and $c_j$ = the OLS coefficient, so soft-threshold by $\lambda/2 = 1$ → (2.0, 0, 0). Ridge shrinks everything proportionally; the lasso shifts by 1 and zeroes the two small ones.</details>
4. Why do we centre $\mathbf X$ and $\mathbf y$ before ridge or lasso, and how is $w_0$ recovered?
   <details><summary>Answer</summary>Because the intercept must not be penalised (it only sets the overall level of $y$). Minimising over $w_0$ gives $w_0 = \bar y - \bar{\mathbf x}^\top\mathbf w$, and substituting turns the problem into ridge/lasso on centred data without an intercept. Afterwards, $w_0 = \bar y - \bar{\mathbf x}^\top\mathbf w$ (slide 59).</details>
5. In coordinate descent for the lasso, feature $j$ has $a_j = \mathbf x_j^\top\mathbf x_j = 50$ and $c_j = \mathbf x_j^\top\mathbf r^{(j)} = 12$. What is the update for $\lambda = 30$? For $\lambda = 20$?
   <details><summary>Answer</summary>Threshold $\lambda/2$. For $\lambda = 30$: $|c_j| = 12 \le 15$, so $w_j = 0$. For $\lambda = 20$: $S(12, 10) = 2$, so $w_j = 2/50 = 0.04$.</details>
6. Code reading: `Lasso(alpha=0.01).fit(X, y)` with $N = 500$ people. Which $\lambda$ of the slides' objective $\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \lambda\lVert\mathbf w\rVert_1$ does this correspond to?
   <details><summary>Answer</summary>scikit-learn minimises $\frac{1}{2N}\lVert\mathbf y-\mathbf X\mathbf w\rVert^2 + \alpha\lVert\mathbf w\rVert_1$; multiplying by $2N$ gives $\lambda = 2N\alpha = 2\cdot500\cdot0.01 = 10$.</details>
7. Why does the $L_1$ penalty produce exact zeros while the $L_2$ penalty does not? Give the geometric and the one-dimensional argument.
   <details><summary>Answer</summary>Geometry: the $L_1$ ball has corners on the coordinate axes, where the expanding error ellipses typically touch it first; the round $L_2$ ball has no corners. One dimension: $|w|$ has a kink at 0 whose slopes range over $[-\lambda, \lambda]$, so the minimum stays at 0 whenever $|c| \le \lambda/2$; $w^2$ is smooth with slope 0 at 0, so ridge gives $c/(a+\lambda)\neq0$ whenever $c\neq0$.</details>
8. The lasso selected SNP rs123 in your GWAS. Give three reasons why rs123 might not be a causal variant.
   <details><summary>Answer</summary>(1) It may be in LD with the true causal variant and selected in its place; (2) population structure or environment may create a non-genetic association (confounding); (3) selection is unstable — it may not be selected in another sample (check bootstrap frequencies / replication). Also: shrinkage biases its effect size, and naive post-selection $p$-values are too optimistic.</details>

## 🧠 Questions & discussion

1. Ridge corresponds to believing that every SNP has a small effect; the lasso to believing most SNPs have none. For height (thousands of tiny effects) and for a Mendelian disease, which would you choose, and why?
2. Slide 54's notes suggest that for a quick estimate one can simply pick a small $\sigma_0^2$ such as 0.1 or 0.01. Why is it better to choose $\lambda$ by validation or cross-validation? When might a value fixed in advance be defensible?
3. The $\lambda$ that predicts best selected 40–60 SNPs while only 10 were causal. If your goal is a list of candidate genes for follow-up experiments, how would you choose $\lambda$ differently?
4. Standardising genotypes gives rare variants a relatively smaller penalty per allele. Is that desirable? Think about how large effects of rare variants are believed to be.
5. The graphical lasso (Module 18) puts an $L_1$ penalty on a precision matrix to learn a sparse gene network. What does a zero mean there, and what caveats from Section 10 carry over?

## 🚀 Bonus — going further

- **Effective degrees of freedom.** For ridge, $\mathrm{df}(\lambda) = \sum_j d_j^2/(d_j^2+\lambda)$ where $d_j$ are the singular values of the centred $\mathbf X$ (`np.linalg.svd`). Recompute the prostate ridge path against $\mathrm{df}(\lambda)$ (from 8 down to 0) to reproduce the layout of ESL Fig. 3.8.
- **The SVD trick for ridge paths.** With $\mathbf X_c = \mathbf U\mathbf D\mathbf V^\top$, $\mathbf w(\lambda) = \mathbf V\,\mathrm{diag}\big(d_j/(d_j^2+\lambda)\big)\mathbf U^\top\mathbf y_c$: one decomposition gives the whole path. Implement it for the 1,000-SNP cohort and time it against the loop of cell 5.2.
- **Adaptive lasso and relaxed lasso.** Refit least squares on the lasso-selected SNPs (relaxed lasso) or weight each SNP's penalty by $1/|\hat w_j^{ridge}|$ (adaptive lasso). Do precision and test MSE improve on the GWAS cohort?
- **Logistic lasso for case–control GWAS** (Wu *et al.* 2009): after Module 09, fit an $L_1$-penalised logistic regression on a simulated case–control phenotype with `sklearn.linear_model.LogisticRegression(solver='liblinear', l1_ratio=1.0, C=...)` (scikit-learn ≥ 1.8; on older versions write `penalty='l1'` instead of `l1_ratio=1.0`).
- **Deep dive in this repository:** [`../../RegularizedLinearRegression/`](../../RegularizedLinearRegression/) applies ridge and lasso (with from-scratch coordinate descent and cross-validation) to correlated gene pathways and to a real wheat genomic-prediction data set with 1,279 markers, and compares them with production genomic-selection methods.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module 07 — Regularization: ridge, lasso and sparse genetic effects     date:
Input: SIMULATED cohorts (50 × 200 SNPs; 1,000 people × 1,000 SNPs in LD blocks, 10 causal); REAL prostate data (97 men)
Output: ridge/lasso coefficients and paths, validation/CV curves, selected SNP sets, bootstrap selection frequencies
Method: MAP with Gaussian prior (ridge, closed form); MAP with Laplace prior (lasso, coordinate descent + soft-thresholding)
Key equation: w = (XᵀX + λI)⁻¹Xᵀy, λ = σ²/σ0²;   w_j ← S(x_jᵀr⁽ʲ⁾, λ/2) / x_jᵀx_j
Key parameters: λ (chosen on validation/CV data), standardisation, warm-start λ grid, scikit-learn α = λ/(2N)
What changed when I modified … (σ0², study size, LD strength, λ choice):
One failure mode: penalising the intercept / unstandardised features / picking a passenger SNP in LD
Connection to 02-620 slides: 002_regression slides 45–70; 015_model-selection--extra slides 1–2
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists*, Spring 2025 lecture slides, decks `002_regression` and `015_model-selection--extra` (course design S. Kim; instructors M. Xu and M. Zhang).
- Hastie T., Tibshirani R., Friedman J. (2009) *The Elements of Statistical Learning*, 2nd ed. Springer — §3.3–3.4 (subset selection, ridge, lasso; Figs. 3.8, 3.10, 3.11; Table 3.4) and §3.8.6 (pathwise coordinate optimisation).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — §7.5 (ridge regression) and Chapter 13 (sparse linear models, lasso, coordinate descent).
- Tibshirani R. (1996) Regression shrinkage and selection via the lasso. *J R Stat Soc B* 58:267–288.
- Hoerl A. E., Kennard R. W. (1970) Ridge regression: biased estimation for nonorthogonal problems. *Technometrics* 12:55–67.
- Friedman J., Hastie T., Höfling H., Tibshirani R. (2007) Pathwise coordinate optimization. *Ann Appl Stat* 1:302–332.
- Wu T. T., Chen Y. F., Hastie T., Sobel E., Lange K. (2009) Genome-wide association analysis by lasso penalized logistic regression. *Bioinformatics* 25:714–721.
- Zou H., Hastie T. (2005) Regularization and variable selection via the elastic net. *J R Stat Soc B* 67:301–320.
- Park T., Casella G. (2008) The Bayesian lasso. *J Am Stat Assoc* 103:681–686.
- Meinshausen N., Bühlmann P. (2010) Stability selection. *J R Stat Soc B* 72:417–473.
- Stamey T. A. *et al.* (1989) Prostate specific antigen in the diagnosis and treatment of adenocarcinoma of the prostate. II. Radical prostatectomy treated patients. *J Urol* 141:1076–1083.

## ➡️ Next

So far the phenotype was a number. In [Module 08 · 🧮 Classification I: k-nearest neighbours and naïve Bayes](08_knn_naive_bayes.ipynb) it becomes a label — cancer or healthy — and the two-step recipe (model, then MLE/MAP) produces our first classifiers.